# 🏥 Breast Cancer AI Screening System — AIMS Study Inspired
## Comprehensive ML Notebook
### Based on: Kelly et al. (2026) *Nature Cancer* & Warren et al. (2026) *Nature Cancer*

---

### 📌 What This Notebook Covers

| Module | Description |
|--------|-------------|
| **1. Environment Setup** | Dependencies, PACS/HL7 integration stubs |
| **2. PACS & HL7 Integration** | DICOM ingestion, HL7 v2/FHIR messaging |
| **3. Data Preprocessing** | DICOM → NumPy, augmentation pipeline |
| **4. Dense Breast Classification** | BIRADS-1 to BIRADS-4 density model |
| **5. Lesion Detection & Classification** | ROI detection, calcification vs mass |
| **6. BIRADS Final Score Classifier** | Full case-level BIRADS scoring |
| **7. Cancer Risk Score** | Operating point calibration, risk stratification |
| **8. Tumor Identification** | Invasive vs In-situ, size, grade prediction |
| **9. Interval Cancer Prediction** | Future cancer detection (next-round model) |
| **10. Fairness & Subgroup Analysis** | Age, ethnicity, IMD, breast density |
| **11. Distribution Shift Monitor** | Drift detection & OP recalibration |
| **12. Full Inference Pipeline** | End-to-end case scoring with report |

---
### 📄 Key Study Statistics Used as Benchmarks
- **Retrospective cohort:** 115,973 mammograms, 5 NHS sites, 39-month follow-up
- **AI Sensitivity (primary):** 0.541 vs 0.437 (first human reader) P<0.001
- **AI Specificity:** 0.943 (noninferior to 0.952, P<0.001)
- **Cancer Detection Rate:** 9.33 vs 7.54 per 1,000 women
- **Interval Cancer Detection:** 25.0% captured by AI
- **Reading time reduction (second-reader replacement):** 32%
- **Prospective:** 9,266 cases, 12 sites, AUC=0.98

## 📦 Section 1 — Environment Setup & Installations

In [ ]:
# UPDATED INSTALLS — adds optuna, onnx, captum
import subprocess, sys
packages = [
    'pydicom','SimpleITK','monai','hl7apy','fhirclient',
    'torch','torchvision','timm','ultralytics','pytorch-grad-cam',
    'opencv-python-headless','albumentations','Pillow',
    'scikit-learn','scikit-image','scipy','lifelines','shap',
    'pandas','numpy','matplotlib','seaborn','tqdm','gradio',
    'optuna','onnx','onnxruntime','captum',
]
failed=[]
for pkg in packages:
    try:
        subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'],
            stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    except subprocess.CalledProcessError:
        failed.append(pkg)
print('Packages installed!' if not failed else f'WARNING: {failed}')


In [ ]:
# SECTION 2 - CORE IMPORTS
# FIX: Added MONAI, timm, ultralytics, pytorch_grad_cam
# FIX: torchvision weights API guard for PyTorch 2.x
import os,sys,json,time,logging,warnings,random
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass,field
from typing import Dict,List,Optional,Tuple,Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from tqdm import tqdm
import torch,torch.nn as nn,torch.nn.functional as F,torch.optim as optim
from torch.utils.data import Dataset,DataLoader,WeightedRandomSampler
import torchvision.transforms as transforms,torchvision.models as tv_models
# FIX: PyTorch 2.x weights API
try:
    from torchvision.models import ResNet50_Weights,DenseNet121_Weights; TORCH2=True
except ImportError:
    TORCH2=False
# timm
try:
    import timm; TIMM_AVAILABLE=True; print(f'  timm {timm.__version__}')
except ImportError:
    TIMM_AVAILABLE=False; print('  WARNING: timm not available')
# MONAI
try:
    import monai
    from monai.transforms import Compose,ScaleIntensity,Resize,RandFlip,RandZoom,NormalizeIntensity
    MONAI_AVAILABLE=True; print(f'  MONAI {monai.__version__}')
except ImportError:
    MONAI_AVAILABLE=False; print('  WARNING: MONAI not available - OpenCV fallback')
# YOLOv8
try:
    from ultralytics import YOLO; YOLO_AVAILABLE=True; print('  YOLOv8 available')
except ImportError:
    YOLO_AVAILABLE=False; print('  WARNING: YOLOv8 not available - stub mode')
# Grad-CAM
try:
    from pytorch_grad_cam import GradCAM,GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE=True; print('  pytorch-grad-cam available')
except ImportError:
    GRADCAM_AVAILABLE=False; print('  WARNING: pytorch-grad-cam not available - manual fallback')
from sklearn.metrics import (roc_auc_score,roc_curve,confusion_matrix,
    classification_report,precision_recall_curve,average_precision_score,brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.preprocessing import label_binarize
from sklearn.model_selection import StratifiedKFold
import cv2
from PIL import Image
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)
SEED=42; random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


In [ ]:
# HELPER - PyTorch 2.x safe model creation
# FIX: tv_models.resnet50(pretrained=True) deprecated in PyTorch 2.x
def _resnet50(pretrained=False):
    if TORCH2:
        w=ResNet50_Weights.DEFAULT if pretrained else None
        return tv_models.resnet50(weights=w)
    return tv_models.resnet50(pretrained=pretrained)
def _densenet121(pretrained=False):
    if TORCH2:
        w=DenseNet121_Weights.DEFAULT if pretrained else None
        return tv_models.densenet121(weights=w)
    return tv_models.densenet121(pretrained=pretrained)
def count_params(m): return sum(p.numel() for p in m.parameters())
print('Model helpers registered (PyTorch 1.x + 2.x compatible)')


## 🏥 Section 2 — PACS & HL7 Integration Layer

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PACS CONFIGURATION
# In production: replace with actual AE titles and PACS endpoints
# Mirrors the SmartBox relay used in the AIMS prospective study
# (time from screen → AI read was 17.7 min median)
# ─────────────────────────────────────────────────────────────────
@dataclass
class PACSConfig:
    """PACS connection config — mirrors AIMS SmartBox/NBSS relay."""
    host: str          = 'localhost'
    port: int          = 4242
    ae_title: str      = 'BREAST_AI'
    calling_ae: str    = 'AIMS_CLIENT'
    timeout: int       = 30
    # DICOM study filters relevant to mammography screening
    modality: str      = 'MG'          # Mammography
    min_image_size: Tuple[int, int] = (3000, 2000)  # AI exclusion criterion (AIMS)
    standard_views: List[str] = field(default_factory=lambda: [
        'MLO_LEFT', 'MLO_RIGHT', 'CC_LEFT', 'CC_RIGHT'   # 4-view standard
    ])

@dataclass  
class HL7Config:
    """HL7 v2.x / FHIR R4 config for NBSS result writeback."""
    hl7_host: str    = 'localhost'
    hl7_port: int    = 2575
    fhir_base: str   = 'http://localhost:8080/fhir'
    sending_app: str = 'AIMS_AI'
    receiving_app: str = 'NBSS'

PACS_CFG = PACSConfig()
HL7_CFG  = HL7Config()
print('✅ PACS & HL7 config loaded')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# PACS INTEGRATION — DICOM INGESTION
# Includes all AI exclusion criteria from AIMS study:
#   - Image size < 3000×2000 pixels → exclude
#   - Missing or extra views (not exactly 4) → exclude
#   - Breast implants → exclude
#   - Technical recalls → exclude
# ─────────────────────────────────────────────────────────────────
try:
    import pydicom
    PYDICOM_AVAILABLE = True
except ImportError:
    PYDICOM_AVAILABLE = False

class PACSClient:
    """Simulated PACS client with AIMS exclusion criteria."""

    # ─── AIMS AI Exclusion Criteria ─────────────────────────────
    EXCLUSION_REASONS = {
        'small_dicom'    : 'Image < 3000×2000 px',
        'missing_views'  : 'Not exactly 4 standard views',
        'extra_views'    : 'Additional non-standard views present',
        'implants'       : 'Breast implants detected',
        'tech_recall'    : 'Technical recall flagged by R1',
        'invalid_dicom'  : 'DICOM file corrupt/invalid',
        'high_risk_screen': 'High/moderate-risk special screening',
    }

    STANDARD_VIEW_TAGS = {
        'CC_LEFT', 'CC_RIGHT', 'MLO_LEFT', 'MLO_RIGHT'
    }

    def __init__(self, config: PACSConfig):
        self.cfg = config
        self._stats = {'processed': 0, 'excluded': 0, 'reasons': {}}

    def load_dicom(self, filepath: str) -> Optional[dict]:
        """Load and validate a single DICOM file."""
        if not PYDICOM_AVAILABLE:
            # Return mock DICOM for demonstration
            return self._mock_dicom(filepath)
        try:
            ds = pydicom.dcmread(filepath)
            return {
                'pixel_array' : ds.pixel_array,
                'patient_id'  : getattr(ds, 'PatientID', 'UNKNOWN'),
                'study_uid'   : getattr(ds, 'StudyInstanceUID', ''),
                'series_uid'  : getattr(ds, 'SeriesInstanceUID', ''),
                'rows'        : getattr(ds, 'Rows', 0),
                'cols'        : getattr(ds, 'Columns', 0),
                'laterality'  : getattr(ds, 'ImageLaterality', ''),
                'view_position': getattr(ds, 'ViewPosition', ''),
                'manufacturer': getattr(ds, 'Manufacturer', ''),
                'model_name'  : getattr(ds, 'ManufacturerModelName', ''),
                'implant_flag': getattr(ds, 'BreastImplantPresent', 'NO'),
                'filepath'    : filepath,
                'raw_ds'      : ds,
            }
        except Exception as e:
            logging.warning(f'DICOM load failed: {filepath} → {e}')
            return None

    def _mock_dicom(self, filepath: str) -> dict:
        """Generate realistic mock DICOM metadata for demonstration."""
        np.random.seed(hash(filepath) % 2**32)
        view = np.random.choice(['CC_LEFT', 'CC_RIGHT', 'MLO_LEFT', 'MLO_RIGHT'])
        mfr  = np.random.choice(['Hologic', 'GE', 'Siemens'],
                                 p=[0.71, 0.07, 0.22])  # AIMS study proportions
        return {
            'pixel_array' : np.random.randint(0, 4096,
                            (3000, 2000), dtype=np.uint16),
            'patient_id'  : f'PT_{abs(hash(filepath)) % 100000:06d}',
            'study_uid'   : f'1.2.3.{abs(hash(filepath)) % 999999}',
            'series_uid'  : f'1.2.4.{abs(hash(filepath)) % 999999}',
            'rows'        : 3000,
            'cols'        : 2000,
            'laterality'  : view.split('_')[1],
            'view_position': view.split('_')[0],
            'manufacturer': mfr,
            'model_name'  : 'Selenia Dimensions' if mfr == 'Hologic' else mfr,
            'implant_flag': 'NO',
            'filepath'    : filepath,
        }

    def validate_case(self, dicom_list: List[dict]) -> Tuple[bool, str]:
        """
        Apply all AIMS exclusion criteria to a 4-view case.
        Returns (is_eligible, reason_if_excluded)
        """
        if not dicom_list:
            return False, 'invalid_dicom'

        # 1. Check exactly 4 standard views
        views = set()
        for d in dicom_list:
            view_key = f"{d.get('view_position','')}_{d.get('laterality','')}"
            views.add(view_key)

        if len(dicom_list) < 4:
            return False, 'missing_views'
        if len(dicom_list) > 4:
            return False, 'extra_views'

        for d in dicom_list:
            # 2. Image size check
            if d['rows'] < self.cfg.min_image_size[0] or \
               d['cols'] < self.cfg.min_image_size[1]:
                return False, 'small_dicom'
            # 3. Implant check
            if d.get('implant_flag', 'NO') == 'YES':
                return False, 'implants'

        return True, 'eligible'

    def get_statistics(self) -> dict:
        return self._stats

# ─── Demonstrate the PACS client ────────────────────────────────
pacs = PACSClient(PACS_CFG)

# Simulate loading 4 views of a case
mock_case = [
    pacs._mock_dicom(f'case001_view{i}.dcm') for i in range(4)
]
# Fix views to be standard
views = ['CC_LEFT', 'CC_RIGHT', 'MLO_LEFT', 'MLO_RIGHT']
for i, (d, v) in enumerate(zip(mock_case, views)):
    d['view_position'] = v.split('_')[0]
    d['laterality']    = v.split('_')[1]

eligible, reason = pacs.validate_case(mock_case)
print(f'✅ Case validation: eligible={eligible}, reason={reason}')
print(f'   Manufacturer: {mock_case[0]["manufacturer"]}')
print(f'   Image size  : {mock_case[0]["rows"]}×{mock_case[0]["cols"]}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# HL7 v2 / FHIR R4 INTEGRATION
# Mimics NBSS writeback described in AIMS prospective study
# At time of study, NBSS lacked AI writeback → this implements it
# ─────────────────────────────────────────────────────────────────
class HL7Messenger:
    """
    HL7 v2.x OBR/OBX message builder for AI screening results.
    Addresses the AIMS finding that NBSS lacked automatic AI writeback.
    """

    def __init__(self, config: HL7Config):
        self.cfg = config
        self._msg_counter = 0

    def build_oru_r01(self, patient_id: str, study_uid: str,
                       ai_result: dict) -> str:
        """
        Build HL7 v2.x ORU^R01 message for AI mammography result.
        OBX segments carry: AI score, BIRADS, recall decision, ROI.
        """
        self._msg_counter += 1
        ts = datetime.now().strftime('%Y%m%d%H%M%S')
        ctrl_id = f'AIMS{self._msg_counter:06d}'

        birads    = ai_result.get('birads', 0)
        recall    = 'RECALL' if ai_result.get('recall', False) else 'NORMAL'
        risk_score = ai_result.get('risk_score', 0.0)
        density   = ai_result.get('density_birads', 'B2')

        msg = (
            f"MSH|^~\\&|{self.cfg.sending_app}||{self.cfg.receiving_app}"
            f"||{ts}||ORU^R01|{ctrl_id}|P|2.5\r"
            f"PID|1||{patient_id}\r"
            f"OBR|1||{study_uid}|MAMMO_SCREEN^Mammography Screening\r"
            # OBX-1: BIRADS Final Score
            f"OBX|1|NM|BIRADS_SCORE^BIRADS Final Score||{birads}||0-6|N|||F\r"
            # OBX-2: AI Recall Decision
            f"OBX|2|ST|AI_RECALL^AI Recall Decision||{recall}|||||F\r"
            # OBX-3: AI Risk Score (0-1)
            f"OBX|3|NM|AI_RISK^AI Cancer Risk Score||{risk_score:.4f}"
            f"||0-1|H|||F\r"
            # OBX-4: Breast Density BIRADS
            f"OBX|4|ST|DENSITY^Breast Density||{density}|||||F\r"
            # OBX-5: Assessment Reason (addresses AIMS gap)
            f"OBX|5|ST|ASSESS_REASON^Assessment Reason"
            f"||AI_PROMPTED^AI System Prompted|||||F\r"
        )

        # Add ROI locations if present
        for idx, roi in enumerate(ai_result.get('rois', []), start=6):
            msg += (
                f"OBX|{idx}|ST|AI_ROI_{idx}^AI Region of Interest {idx-5}"
                f"||L:{roi['laterality']};V:{roi['view']};"
                f"X:{roi.get('x',0)};Y:{roi.get('y',0)};"
                f"W:{roi.get('w',0)};H:{roi.get('h',0)}"
                f"|||||F\r"
            )
        return msg

    def build_fhir_diagnostic_report(self, patient_id: str,
                                      ai_result: dict) -> dict:
        """Build FHIR R4 DiagnosticReport resource for AI result."""
        return {
            'resourceType' : 'DiagnosticReport',
            'status'       : 'final',
            'category'     : [{'coding': [{
                'system' : 'http://loinc.org',
                'code'   : '36643-5',
                'display': 'Breast - bilateral Mammogram'
            }]}],
            'code'         : {'text': 'AI Mammography Screening Result'},
            'subject'      : {'reference': f'Patient/{patient_id}'},
            'issued'       : datetime.now().isoformat() + 'Z',
            'performer'    : [{'display': 'AIMS AI System v1.2 (Google)'}],
            'result'       : [
                {'display': f"BIRADS: {ai_result.get('birads', 0)}"},
                {'display': f"Risk Score: {ai_result.get('risk_score', 0):.4f}"},
                {'display': f"Density: {ai_result.get('density_birads', 'B2')}"},
                {'display': f"Recall: {ai_result.get('recall', False)}"},
            ],
            'conclusion'   : (
                f"AI BIRADS {ai_result.get('birads',0)}: "
                + ('RECALL recommended' if ai_result.get('recall') else 'NORMAL — routine follow-up')
            )
        }

# ─── Demo ────────────────────────────────────────────────────────
hl7 = HL7Messenger(HL7_CFG)
sample_result = {
    'birads'        : 4,
    'recall'        : True,
    'risk_score'    : 0.73,
    'density_birads': 'B3',
    'rois'          : [{'laterality': 'L', 'view': 'MLO', 'x': 512, 'y': 320, 'w': 64, 'h': 64}]
}
hl7_msg = hl7.build_oru_r01('PT_000123', '1.2.3.456789', sample_result)
fhir_rpt = hl7.build_fhir_diagnostic_report('PT_000123', sample_result)

print('📨 HL7 ORU^R01 Message (first 400 chars):') 
print(hl7_msg[:400])
print('\n📋 FHIR DiagnosticReport conclusion:')
print(fhir_rpt['conclusion'])

## 🖼️ Section 3 — Data Preprocessing & Augmentation

In [ ]:
# SECTION 4 - MONAI PREPROCESSING + DATASET
# FIX: df_all created HERE (was used before definition in training cell)
# FIX: MONAI preprocessing pipeline added
# FIX: transforms.RandomAdjustSharpness version guard

def simulate_aims_dataframe(n=2000,seed=42):
    rng=np.random.default_rng(seed)
    n_c=int(n*0.017); cl=np.array([1]*n_c+[0]*(n-n_c)); rng.shuffle(cl)
    pi=np.where(cl==1)[0]; tp=np.zeros(n,dtype=int)
    tp[pi]=rng.choice([1,2,3],size=len(pi),p=[0.437,0.168,0.394])
    return pd.DataFrame({
        'patient_id':[f'PT_{i:06d}' for i in range(n)],
        'age':rng.integers(50,71,n),
        'cancer_label':cl.astype(int),'cancer_timepoint':tp,
        'density_birads':rng.choice([1,2,3,4],n,p=[0.10,0.40,0.40,0.10]),
        'birads_score':rng.choice([1,2,3,4,5,6],n,p=[0.70,0.15,0.08,0.04,0.02,0.01]),
        'invasive':[int(rng.random()<0.80) if c else 0 for c in cl],
        'lesion_size_mm':[max(0.,float(rng.normal(15,8))) if c else 0. for c in cl],
        'imd_decile':rng.integers(1,11,n),
        'ethnicity':rng.choice(['White','Black','Asian','Mixed','Other','Unknown'],
            n,p=[0.60,0.09,0.15,0.03,0.04,0.09]),
        'manufacturer':rng.choice(['Hologic','Siemens','GE'],n,p=[0.71,0.22,0.07]),
        'screen_type':rng.choice(['prevalent','incident'],n,p=[0.139,0.861]),
        'risk_score':np.clip(
            [float(rng.normal(0.7,0.2)) if c else float(rng.normal(0.1,0.15)) for c in cl],0,1),
        'split':rng.choice(['train','val','test'],n,p=[0.70,0.15,0.15]),
    })

df_all=simulate_aims_dataframe(n=2000)
df_train=df_all[df_all.split=='train'].copy()
df_val=df_all[df_all.split=='val'].copy()
df_test=df_all[df_all.split=='test'].copy()
print(f'Dataset: {len(df_all):,} | cancer={df_all.cancer_label.mean()*100:.1f}%')
print(f'Splits: train={len(df_train)} val={len(df_val)} test={len(df_test)}')

class MONAIPreprocessor:
    TARGET_SHAPE=(224,224)
    VENDOR_CLAHE={'Hologic':{'clip_limit':2.0,'tile_grid_size':(8,8)},
                  'GE':{'clip_limit':1.5,'tile_grid_size':(8,8)},
                  'Siemens':{'clip_limit':2.5,'tile_grid_size':(8,8)},
                  'default':{'clip_limit':2.0,'tile_grid_size':(8,8)}}
    def __init__(self,vendor='Hologic',augment=False):
        p=self.VENDOR_CLAHE.get(vendor,self.VENDOR_CLAHE['default'])
        self.clahe=cv2.createCLAHE(**p)
        if MONAI_AVAILABLE:
            aug_t=[RandFlip(spatial_axis=1,prob=0.5)] if augment else []
            self.pipeline=Compose([ScaleIntensity(),Resize(self.TARGET_SHAPE),*aug_t,
                NormalizeIntensity(subtrahend=[0.485,0.456,0.406],
                                   divisor=[0.229,0.224,0.225],channel_wise=True)])
        else:
            self.pipeline=None
    def normalize_16bit(self,img):
        img=img.astype(np.float32); p2,p98=np.percentile(img,[2,98])
        return np.clip((img-p2)/(p98-p2+1e-8),0,1)
    def apply_clahe(self,img):
        return self.clahe.apply((img*255).astype(np.uint8)).astype(np.float32)/255.
    def remove_pectoral(self,img,view='MLO'):
        if 'MLO' not in view.upper(): return img
        h,w=img.shape[:2]; mask=np.ones_like(img)
        for row in range(min(h//3,200)):
            col=int(row*0.8)
            if col<w: mask[row,:col]=0.3
        return img*mask
    def preprocess_single(self,pixel_array,view='CC'):
        img=self.normalize_16bit(pixel_array)
        img=self.apply_clahe(img)
        img=self.remove_pectoral(img,view)
        img_3ch=np.stack([img,img,img],axis=0)
        if self.pipeline is not None:
            out=self.pipeline(img_3ch)
            return (out.numpy() if hasattr(out,'numpy') else out).astype(np.float32)
        img=cv2.resize(img,(self.TARGET_SHAPE[1],self.TARGET_SHAPE[0]))
        img_3ch=np.stack([img,img,img],axis=0)
        m=np.array([0.485,0.456,0.406])[:,None,None]
        s=np.array([0.229,0.224,0.225])[:,None,None]
        return ((img_3ch-m)/s).astype(np.float32)
    def preprocess_case(self,dicom_list):
        vo=['CC_LEFT','CC_RIGHT','MLO_LEFT','MLO_RIGHT']
        vm={f"{d['view_position']}_{d['laterality']}":d for d in dicom_list}
        out=[]
        for v in vo:
            d=vm.get(v)
            arr=self.preprocess_single(d['pixel_array'],d['view_position']) if d \
                else np.zeros((3,*self.TARGET_SHAPE),dtype=np.float32)
            out.append(arr)
        return np.stack(out,axis=0)

class MammographyDataset(Dataset):
    def __init__(self,df,vendor='Hologic',augment=False):
        self.df=df.reset_index(drop=True)
        self.prep=MONAIPreprocessor(vendor=vendor,augment=augment)
        aug_list=[transforms.RandomHorizontalFlip(p=0.5)]
        try: aug_list.append(transforms.RandomAdjustSharpness(2,p=0.3))  # guard torchvision<0.9
        except AttributeError: pass
        self.aug=transforms.Compose(aug_list) if augment else None
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]
        images=torch.randn(4,3,224,224)  # Demo: replace with real DICOM load
        if self.aug: images=torch.stack([self.aug(images[i]) for i in range(4)])
        labels={
            'cancer':torch.tensor(int(row['cancer_label']),dtype=torch.long),
            'density_birads':torch.tensor(int(row['density_birads'])-1,dtype=torch.long),
            'birads_score':torch.tensor(int(row['birads_score']),dtype=torch.long),
            'cancer_tp':torch.tensor(int(row['cancer_timepoint']),dtype=torch.long),
            'invasive':torch.tensor(float(row['invasive']),dtype=torch.float),
            'lesion_size':torch.tensor(float(row['lesion_size_mm']),dtype=torch.float),
            'risk_score':torch.tensor(float(row['risk_score']),dtype=torch.float),
            'patient_id':str(row['patient_id']),
        }
        return images,labels

print(f'Preprocessor: {"MONAI" if MONAI_AVAILABLE else "OpenCV fallback"}')
print('MammographyDataset defined')


## 🔬 Section 3B — CLAHE Preprocessing Upgrade + mAP50 Evaluator

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3B — CLAHE UPGRADE + mAP50 EVALUATION
# Based on: Abdikenov et al. J.Imaging 2025 — key finding:
#   CLAHE (clip=2.0, tileGrid=8×8) alone improved mAP50 by up to +0.209
#   (INbreast: 0.754 → 0.963)
# mAP50 = mean Average Precision at IoU threshold 0.50
#   standard object-detection metric alongside AUC/sensitivity
# ─────────────────────────────────────────────────────────────────────────────

# ── PART A: Standalone CLAHE pipeline (replaces basic cv2 call in MONAIPreprocessor) ──

class CLAHEPipeline:
    """
    Full CLAHE + breast-cropping pipeline from Abdikenov et al.
    Plug-in replacement for the apply_clahe() method in MONAIPreprocessor.
    Steps:
      1. 16-bit → float normalisation
      2. Automated breast-region cropping (column scan algorithm)
      3. CLAHE on luminance channel (8×8 tiles, clip limit 2.0)
      4. Bounding-box coordinate adjustment after crop
    """

    def __init__(self, clip_limit: float = 2.0, tile_grid: tuple = (8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid  = tile_grid
        self.clahe      = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)

    # ── Step 1: normalise 16-bit pixel values to [0,1] float ──
    def _normalize(self, img: np.ndarray) -> np.ndarray:
        img = img.astype(np.float32)
        p2, p98 = np.percentile(img, [2, 98])
        return np.clip((img - p2) / (p98 - p2 + 1e-8), 0, 1)

    # ── Step 2: automated breast-region crop (Abdikenov 2025 algorithm) ──
    def _crop_breast(self, img: np.ndarray, offset_px: int = 200) -> tuple:
        """
        Scans a central vertical strip (20% of height).
        Finds the breast boundary by looking for the first column
        whose mean intensity exceeds threshold=30 (on 0-255 scale).
        Returns (cropped_image, cropx_min, cropx_max).
        """
        h, w = img.shape[:2]
        strip_h  = h // 5                        # 20% of height
        y_start  = h // 2 - strip_h // 2
        y_end    = y_start + strip_h
        threshold = 30.0 / 255.0                 # normalised threshold

        # Compute mean intensity in 50-px-wide edge columns
        left_mean  = img[y_start:y_end, :50].mean()
        right_mean = img[y_start:y_end, -50:].mean()

        if left_mean < right_mean:               # breast is on the right
            crop_dir = 'left'
            for col in range(w):
                if img[y_start:y_end, col].mean() > threshold:
                    cropx_min = max(0, col - offset_px)
                    cropx_max = w
                    break
            else:
                cropx_min, cropx_max = 0, w
        else:                                    # breast is on the left
            crop_dir = 'right'
            for col in range(w - 1, -1, -1):
                if img[y_start:y_end, col].mean() > threshold:
                    cropx_min = 0
                    cropx_max = min(w, col + offset_px)
                    break
            else:
                cropx_min, cropx_max = 0, w

        cropped = img[:, cropx_min:cropx_max]
        return cropped, cropx_min, cropx_max

    # ── Step 3: CLAHE on grayscale or luminance channel ──
    def _apply_clahe(self, img: np.ndarray) -> np.ndarray:
        img_u8 = (img * 255).astype(np.uint8)
        if img_u8.ndim == 2:                     # grayscale
            return self.clahe.apply(img_u8).astype(np.float32) / 255.0
        else:                                    # colour: apply to Y channel
            yuv = cv2.cvtColor(img_u8, cv2.COLOR_BGR2YUV)
            yuv[:, :, 0] = self.clahe.apply(yuv[:, :, 0])
            return cv2.cvtColor(yuv, cv2.COLOR_YUV2BGR).astype(np.float32) / 255.0

    # ── Step 4: adjust YOLO bounding boxes after crop ──
    @staticmethod
    def adjust_boxes_after_crop(boxes_yolo: list, cropx_min: int,
                                 cropx_max: int, orig_w: int) -> list:
        """
        boxes_yolo: list of [cx, cy, bw, bh] in normalised [0,1] coords
        Returns adjusted boxes clipped to [0,1] relative to cropped width.
        """
        new_w   = cropx_max - cropx_min
        adjusted = []
        for (cx, cy, bw, bh) in boxes_yolo:
            # Convert to absolute pixel coords
            cx_abs = cx * orig_w
            bw_abs = bw * orig_w
            # Shift relative to crop
            cx_new = (cx_abs - cropx_min) / new_w
            bw_new = bw_abs / new_w
            # Clamp to [0,1]
            cx_new = max(0.0, min(1.0, cx_new))
            bw_new = max(0.0, min(1.0, bw_new))
            if 0.0 < cx_new < 1.0 and bw_new > 0:
                adjusted.append((cx_new, cy, bw_new, bh))
        return adjusted

    # ── Main entry: full pipeline on a single image ──
    def process(self, pixel_array: np.ndarray,
                 boxes_yolo: list = None,
                 view: str = 'CC') -> dict:
        """
        Args:
            pixel_array : H×W uint16 or float32 mammogram
            boxes_yolo  : optional list of [cx,cy,bw,bh] normalised GT boxes
            view        : 'CC' or 'MLO'
        Returns dict with keys:
            'image'     : H×W float32 processed image
            'boxes'     : adjusted bounding boxes (if provided)
            'cropx_min' : crop left boundary (pixels)
            'cropx_max' : crop right boundary (pixels)
        """
        orig_w = pixel_array.shape[1]
        img    = self._normalize(pixel_array)
        img, cropx_min, cropx_max = self._crop_breast(img)
        img    = self._apply_clahe(img)

        result = {'image': img, 'cropx_min': cropx_min, 'cropx_max': cropx_max}

        if boxes_yolo is not None:
            result['boxes'] = self.adjust_boxes_after_crop(
                boxes_yolo, cropx_min, cropx_max, orig_w)
        return result


# ── PART B: mAP50 evaluator ──

class MAP50Evaluator:
    """
    Mean Average Precision at IoU=0.50 (COCO-style).
    Used by Abdikenov et al. as primary detection metric:
      YOLOv12-L mass mAP50 = 0.963 (INbreast, with CLAHE)
      RTMDet-X  mass mAP50 = 0.697 (combined dataset, with TL)

    Works per-class; call add_image() for every image in val/test set,
    then compute() to get per-class AP and overall mAP50.
    """

    IOU_THRESHOLD = 0.50

    def __init__(self, class_names: list = None):
        self.class_names = class_names or ['Mass', 'Calcification', 'Distortion', 'Asymmetry']
        self._reset()

    def _reset(self):
        # per class: list of (confidence, tp_flag)
        self._detections = {c: [] for c in self.class_names}
        self._n_gt       = {c: 0  for c in self.class_names}

    @staticmethod
    def _iou(a: tuple, b: tuple) -> float:
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b
        ix1, iy1 = max(ax1, bx1), max(ay1, by1)
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        if ix2 <= ix1 or iy2 <= iy1:
            return 0.0
        inter = (ix2 - ix1) * (iy2 - iy1)
        union = (ax2-ax1)*(ay2-ay1) + (bx2-bx1)*(by2-by1) - inter
        return inter / (union + 1e-8)

    def add_image(self, predictions: list, ground_truths: list):
        """
        predictions : list of {'bbox':(x1,y1,x2,y2), 'confidence':float, 'class':str}
        ground_truths: list of {'bbox':(x1,y1,x2,y2), 'class':str}
        """
        # Count GT per class
        for gt in ground_truths:
            c = gt['class']
            if c in self._n_gt:
                self._n_gt[c] += 1

        # Match predictions to GT
        matched_gt = set()
        # Sort by confidence descending
        preds_sorted = sorted(predictions, key=lambda x: -x['confidence'])

        for pred in preds_sorted:
            pc = pred['class']
            if pc not in self._detections:
                continue
            best_iou, best_gi = 0.0, -1
            for gi, gt in enumerate(ground_truths):
                if gt['class'] != pc or gi in matched_gt:
                    continue
                iou = self._iou(pred['bbox'], gt['bbox'])
                if iou > best_iou:
                    best_iou, best_gi = iou, gi
            is_tp = best_iou >= self.IOU_THRESHOLD
            if is_tp:
                matched_gt.add(best_gi)
            self._detections[pc].append((pred['confidence'], int(is_tp)))

    def _average_precision(self, class_name: str) -> float:
        dets   = sorted(self._detections[class_name], key=lambda x: -x[0])
        n_gt   = self._n_gt[class_name]
        if n_gt == 0:
            return 0.0
        tp_cum, fp_cum = 0, 0
        precs, recs = [], []
        for _, tp_flag in dets:
            if tp_flag:
                tp_cum += 1
            else:
                fp_cum += 1
            precs.append(tp_cum / (tp_cum + fp_cum))
            recs.append(tp_cum / n_gt)
        # 11-point interpolation
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            p = max([p for p, r in zip(precs, recs) if r >= t], default=0.0)
            ap += p / 11.0
        return ap

    def compute(self, verbose: bool = True) -> dict:
        aps = {c: self._average_precision(c) for c in self.class_names}
        valid_aps = [v for v in aps.values() if self._n_gt[c] > 0 or True]
        map50 = np.mean([v for v in aps.values()])
        result = {'per_class_AP': aps, 'mAP50': map50}
        if verbose:
            print(f'\n── mAP50 Results (IoU=0.50) ──')
            for c, ap in aps.items():
                n = self._n_gt[c]
                print(f'  {c:20s}  AP={ap:.3f}  GT={n}')
            print(f'  {"─"*42}')
            print(f'  {"mAP50":20s}  {map50:.3f}')
            print(f'\n  Benchmark (Abdikenov 2025):')
            print(f'  Mass mAP50 (INbreast+TL) = 0.995')
            print(f'  Mass mAP50 (VinDr-Mammo) = 0.590')
            print(f'  Calc mAP50 (all datasets) < 0.116  ← hardest class')
        self._reset()
        return result


# ── DEMO: CLAHE pipeline ──
clahe_pipe = CLAHEPipeline(clip_limit=2.0, tile_grid=(8, 8))
dummy_mammo = np.random.randint(0, 4096, (3000, 2000), dtype=np.uint16)
dummy_boxes = [(0.45, 0.50, 0.10, 0.12), (0.30, 0.35, 0.06, 0.08)]  # [cx,cy,bw,bh]
result = clahe_pipe.process(dummy_mammo, boxes_yolo=dummy_boxes, view='MLO')
print(f'CLAHE pipeline:')
print(f'  Input  : {dummy_mammo.shape} uint16')
print(f'  Output : {result["image"].shape} float32')
print(f'  Crop   : x=[{result["cropx_min"]} → {result["cropx_max"]}]')
print(f'  Boxes before crop: {len(dummy_boxes)}')
print(f'  Boxes after  crop: {len(result["boxes"])}')

# ── DEMO: mAP50 evaluator ──
evaluator = MAP50Evaluator(class_names=['Mass', 'Calcification'])
rng = np.random.default_rng(42)
for _ in range(50):   # simulate 50 images
    preds = [{'bbox': (100, 100, 200, 200), 'confidence': float(rng.uniform(0.3,0.95)), 'class': 'Mass'}]
    gts   = [{'bbox': (105, 105, 195, 195), 'class': 'Mass'}]  if rng.random() > 0.3 else []
    evaluator.add_image(preds, gts)

metrics = evaluator.compute(verbose=True)
print(f'\n✅  CLAHE + mAP50 modules ready.')
print(f'    Plug CLAHEPipeline.process() into MammographyDataset.__getitem__()')
print(f'    Plug MAP50Evaluator into your validation loop alongside AUC.')


## 🫁 Section 4 — Dense Breast Classification (BIRADS Density)

In [ ]:
# DENSITY CLASSIFIER (EfficientNet-B3 + cross-view attention)
# FIX: pretrained=True -> use _resnet50/_densenet121 helpers (PyTorch 2.x safe)
# FIX: nn.MultiheadAttention batch_first guard for PyTorch < 1.9

class DensityClassifier(nn.Module):
    BIRADS_LABELS={0:'BIRADS-1 (Almost entirely fatty)',1:'BIRADS-2 (Scattered fibroglandular)',
                   2:'BIRADS-3 (Heterogeneously dense)',3:'BIRADS-4 (Extremely dense)'}
    def __init__(self,pretrained=False):
        super().__init__()
        if TIMM_AVAILABLE:
            self.backbone=timm.create_model('efficientnet_b3',pretrained=pretrained,
                                             num_classes=0,global_pool='avg'); feat_dim=1536
        else:
            base=_resnet50(pretrained)  # FIX: PyTorch 2.x safe
            self.backbone=nn.Sequential(*list(base.children())[:-1],nn.Flatten()); feat_dim=2048
        self._attn_old=False
        try:
            self.view_attn=nn.MultiheadAttention(feat_dim,num_heads=8,batch_first=True)
        except TypeError:  # FIX: PyTorch < 1.9 does not have batch_first
            self.view_attn=nn.MultiheadAttention(feat_dim,num_heads=8); self._attn_old=True
        self.classifier=nn.Sequential(nn.LayerNorm(feat_dim),nn.Dropout(0.3),
            nn.Linear(feat_dim,256),nn.GELU(),nn.Dropout(0.2),nn.Linear(256,4))
    def forward(self,x):
        B,V,C,H,W=x.shape
        feats=self.backbone(x.view(B*V,C,H,W)).view(B,V,-1)
        if self._attn_old:
            ft=feats.permute(1,0,2); ao,_=self.view_attn(ft,ft,ft); cf=ao.mean(0)
        else:
            ao,_=self.view_attn(feats,feats,feats); cf=ao.mean(1)
        lg=self.classifier(cf); pr=torch.softmax(lg,dim=-1)
        return {'logits':lg,'probs':pr,'pred':torch.argmax(pr,dim=-1),
                'density_score':(pr*torch.arange(4,device=pr.device).float()).sum(1)}

density_model=DensityClassifier(pretrained=False).to(DEVICE)
print(f'DensityClassifier params: {count_params(density_model):,}')
with torch.no_grad():
    dummy=torch.randn(2,4,3,224,224).to(DEVICE)
    out=density_model(dummy)
print(f'  preds: {out["pred"].tolist()}')
print(f'  labels: {[DensityClassifier.BIRADS_LABELS[p] for p in out["pred"].tolist()]}')


## 🔬 Section 5 — Lesion Detection & Classification

In [ ]:
# LESION DETECTION: DenseNet121 multi-task ROI classifier + YOLOv8 detector
# FIX: pretrained=True deprecated -> use _resnet50() helper
# NEW: YOLOv8 bounding-box detector (was entirely missing from original)

class LesionClassifier(nn.Module):
    LESION_TYPES=['None','Mass','Calcification','Distortion','Asymmetry']
    GRADES=['G1','G2','G3','No_grade']
    def __init__(self,pretrained=False):
        super().__init__()
        base=_resnet50(pretrained)  # FIX: PyTorch 2.x safe
        fd=base.fc.in_features
        self.backbone=nn.Sequential(*list(base.children())[:-1],nn.Flatten())
        self.trunk=nn.Sequential(nn.Linear(fd,512),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(512,256),nn.GELU(),nn.Dropout(0.2))
        self.head_lesion_type=nn.Linear(256,len(self.LESION_TYPES))
        self.head_malignancy=nn.Linear(256,1)
        self.head_invasive=nn.Linear(256,1)
        self.head_grade=nn.Linear(256,len(self.GRADES))
        self.head_size_mm=nn.Linear(256,1)
    def forward(self,roi):
        s=self.trunk(self.backbone(roi))
        return {'lesion_type_logits':self.head_lesion_type(s),
                'malignancy_prob':torch.sigmoid(self.head_malignancy(s)).squeeze(1),
                'invasive_prob':torch.sigmoid(self.head_invasive(s)).squeeze(1),
                'grade_logits':self.head_grade(s),
                'size_mm_pred':F.relu(self.head_size_mm(s)).squeeze(1)}

class YOLOLesionDetector:
    LESION_CLASSES=['Mass','Calcification','Distortion','Asymmetry']
    IOU_THRESHOLD=0.1  # AIMS paper threshold
    def __init__(self,model_path=None,confidence=0.25,iou=0.45):
        self.confidence=confidence; self.iou=iou; self.model=None
        if YOLO_AVAILABLE and model_path and Path(model_path).exists():
            self.model=YOLO(model_path); print(f'YOLOv8 loaded: {model_path}')
        else:
            print('YOLOv8 STUB mode - train on CBIS-DDSM/VinDr-Mammo for production')
    def detect(self,image_np,view='CC',laterality='L'):
        if self.model is not None:
            results=self.model(image_np,conf=self.confidence,iou=self.iou,verbose=False)
            dets=[]
            for r in results:
                for box in r.boxes:
                    x1,y1,x2,y2=box.xyxy[0].tolist(); cls=int(box.cls[0].item())
                    dets.append({'bbox':(int(x1),int(y1),int(x2),int(y2)),
                        'confidence':float(box.conf[0].item()),
                        'lesion_class':self.LESION_CLASSES[cls%len(self.LESION_CLASSES)],
                        'view':view,'laterality':laterality})
            return dets
        return self._stub(image_np,view,laterality)
    def _stub(self,img,view,lat):
        h=img.shape[0] if hasattr(img,'shape') else 224
        w=img.shape[1] if hasattr(img,'shape') else 224
        rng=np.random.default_rng(42)
        if rng.random()>0.3: return []
        x1=int(w*rng.uniform(.2,.6)); y1=int(h*rng.uniform(.2,.6))
        bw=int(w*rng.uniform(.05,.15)); bh=int(h*rng.uniform(.05,.15))
        return [{'bbox':(x1,y1,x1+bw,y1+bh),'confidence':float(rng.uniform(.4,.9)),
                 'lesion_class':str(rng.choice(self.LESION_CLASSES)),
                 'view':view,'laterality':lat,'stub':True}]
    @staticmethod
    def compute_iou(a,b):
        ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
        ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
        if ix2<=ix1 or iy2<=iy1: return 0.
        inter=(ix2-ix1)*(iy2-iy1)
        return inter/((ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter+1e-8)
    def localization_sensitivity(self,pred_boxes,gt_boxes):
        if not gt_boxes: return {'sensitivity':None,'tp':0,'fn':0,'fp':len(pred_boxes)}
        hits=0; matched=set()
        for pb in pred_boxes:
            for gi,gb in enumerate(gt_boxes):
                if gi not in matched and self.compute_iou(pb,gb)>=self.IOU_THRESHOLD:
                    hits+=1; matched.add(gi); break
        tp=hits; fn=len(gt_boxes)-hits; fp=max(0,len(pred_boxes)-hits)
        return {'sensitivity':tp/max(tp+fn,1),'tp':tp,'fn':fn,'fp':fp}

lesion_model=LesionClassifier(pretrained=False).to(DEVICE)
yolo_detector=YOLOLesionDetector()
print(f'LesionClassifier params: {count_params(lesion_model):,}')
with torch.no_grad():
    rd=torch.randn(3,3,224,224).to(DEVICE); lo=lesion_model(rd)
print('LesionClassifier outputs:')
for k,v in lo.items(): print(f'  {k}: {tuple(v.shape)}')
d=yolo_detector.detect(np.zeros((224,224,3),dtype=np.uint8))
print(f'YOLOv8 stub detections: {len(d)}')


## 🧬 Section 5B — Microcalcification Patch Classifier

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5B — MICROCALCIFICATION PATCH CLASSIFIER
# Why separate from the main lesion detector:
#   Calcifications are 0.1–1mm — downscaling to 640×640 destroys them.
#   Solution: tile the full-resolution image into 256×256 patches,
#   run a lightweight EfficientNet-B0 on each tile.
#   Abdikenov et al.: mass mAP50=0.963, calc mAP50<0.116 with shared model.
#   Dedicated patch classifier is the standard approach for DCIS detection.
# ─────────────────────────────────────────────────────────────────────────────

class PatchTiler:
    """
    Tiles a mammogram into overlapping 256×256 patches.
    Overlap prevents calcification clusters at tile boundaries being missed.
    """

    def __init__(self, patch_size: int = 256, stride: int = 192):
        """stride < patch_size creates overlap. 192 → 25% overlap."""
        self.patch_size = patch_size
        self.stride     = stride

    def extract(self, image: np.ndarray) -> list:
        """
        Args:
            image: H×W float32 mammogram (already CLAHE-processed)
        Returns:
            list of {'patch': np.ndarray (256,256), 'row': int, 'col': int}
        """
        h, w    = image.shape[:2]
        patches = []
        for r in range(0, h - self.patch_size + 1, self.stride):
            for c in range(0, w - self.patch_size + 1, self.stride):
                patch = image[r:r+self.patch_size, c:c+self.patch_size]
                patches.append({'patch': patch, 'row': r, 'col': c})
        return patches

    def patch_to_tensor(self, patch: np.ndarray) -> torch.Tensor:
        """Convert H×W float32 → 1×3×256×256 tensor (ImageNet normalisation)."""
        p3 = np.stack([patch, patch, patch], axis=0).astype(np.float32)
        t  = torch.from_numpy(p3).unsqueeze(0)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        return (t - mean) / std


class CalcificationPatchClassifier(nn.Module):
    """
    Lightweight EfficientNet-B0 binary patch classifier.
    Label: 1 = patch contains calcification cluster, 0 = negative.

    Training note: requires patches labelled from CBIS-DDSM or INbreast
    calcification bounding boxes — see prompt 03_calc_training for full setup.

    At inference: run on all patches, keep patches with prob > threshold,
    merge adjacent hot patches → final cluster bounding boxes.
    """

    def __init__(self, pretrained: bool = False):
        super().__init__()
        if TIMM_AVAILABLE:
            self.backbone = timm.create_model(
                'efficientnet_b0', pretrained=pretrained,
                num_classes=0, global_pool='avg'
            )
            feat_dim = 1280
        else:
            base = _resnet50(pretrained)
            self.backbone = nn.Sequential(*list(base.children())[:-1], nn.Flatten())
            feat_dim = 2048

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)          # binary: calc present / absent
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Returns sigmoid probability of calcification presence."""
        return torch.sigmoid(self.classifier(self.backbone(x))).squeeze(1)


class CalcificationInferencePipeline:
    """
    Full inference pipeline:
      mammogram → tile → classify each patch → merge hot patches → ROI list
    """

    def __init__(self, model: CalcificationPatchClassifier,
                 tiler: PatchTiler = None,
                 threshold: float = 0.50,
                 device: torch.device = DEVICE):
        self.model     = model.eval().to(device)
        self.tiler     = tiler or PatchTiler(patch_size=256, stride=192)
        self.threshold = threshold
        self.device    = device

    @torch.no_grad()
    def predict(self, mammogram: np.ndarray) -> dict:
        """
        Args:
            mammogram: H×W float32 (CLAHE-processed)
        Returns:
            {'hot_patches': list, 'cluster_rois': list, 'case_score': float}
        """
        patches    = self.tiler.extract(mammogram)
        hot_patches = []

        for p_info in patches:
            tensor = self.tiler.patch_to_tensor(p_info['patch']).to(self.device)
            prob   = float(self.model(tensor).item())
            if prob >= self.threshold:
                hot_patches.append({
                    'row': p_info['row'],
                    'col': p_info['col'],
                    'prob': prob,
                    'bbox': (p_info['col'], p_info['row'],
                             p_info['col'] + self.tiler.patch_size,
                             p_info['row'] + self.tiler.patch_size),
                })

        cluster_rois = self._merge_patches(hot_patches)
        case_score   = max((p['prob'] for p in hot_patches), default=0.0)

        return {
            'hot_patches' : hot_patches,
            'cluster_rois': cluster_rois,
            'case_score'  : case_score,
            'n_patches'   : len(patches),
            'n_hot'       : len(hot_patches),
        }

    @staticmethod
    def _merge_patches(hot_patches: list, merge_gap: int = 64) -> list:
        """
        Merge spatially adjacent hot patches into cluster bounding boxes.
        Two patches are merged if their centres are within merge_gap pixels.
        """
        if not hot_patches:
            return []
        merged, used = [], set()
        for i, pa in enumerate(hot_patches):
            if i in used:
                continue
            cluster = [pa]
            used.add(i)
            for j, pb in enumerate(hot_patches):
                if j in used:
                    continue
                dist = ((pa['row'] - pb['row'])**2 +
                        (pa['col'] - pb['col'])**2) ** 0.5
                if dist <= merge_gap:
                    cluster.append(pb)
                    used.add(j)
            x1 = min(p['bbox'][0] for p in cluster)
            y1 = min(p['bbox'][1] for p in cluster)
            x2 = max(p['bbox'][2] for p in cluster)
            y2 = max(p['bbox'][3] for p in cluster)
            merged.append({
                'bbox'      : (x1, y1, x2, y2),
                'max_prob'  : max(p['prob'] for p in cluster),
                'n_patches' : len(cluster),
            })
        return merged


# ── DEMO ──
calc_model    = CalcificationPatchClassifier(pretrained=False).to(DEVICE)
calc_pipeline = CalcificationInferencePipeline(calc_model, threshold=0.50)

# Simulate a CLAHE-processed 3000×2000 mammogram
dummy_mammo_f32 = np.random.rand(3000, 2000).astype(np.float32)

# Count tiles
tiler   = PatchTiler(patch_size=256, stride=192)
patches = tiler.extract(dummy_mammo_f32)
print(f'Calcification patch classifier:')
print(f'  Model params    : {count_params(calc_model):,}')
print(f'  Patch size      : 256×256 px  (stride=192 → 25% overlap)')
print(f'  Patches per scan: {len(patches)}  (from 3000×2000 image)')

# Run stub inference on a small crop for speed
small_demo  = np.random.rand(512, 512).astype(np.float32)
demo_result = calc_pipeline.predict(small_demo)
print(f'  Demo (512×512)  : {demo_result["n_patches"]} patches, '
      f'{demo_result["n_hot"]} hot, '
      f'{len(demo_result["cluster_rois"])} clusters')
print(f'  Case score      : {demo_result["case_score"]:.3f}')
print()
print('⚠  To get real performance:')
print('   1. Train on CBIS-DDSM calcification bounding boxes')
print('   2. See prompt → 03_calcification_patch_training')
print('   3. Load weights: calc_model.load_state_dict(torch.load("calc_patch.pth"))')


## 🏷️ Section 6 — BIRADS Final Score Classification

In [ ]:
# ENSEMBLE CLASSIFIER: ViT-B/16 + DenseNet121 + EfficientNet-B3
# NEW: Replaces single ResNet BIRADSScoringModel with proper 3-backbone ensemble
# FIX: All pretrained= args use PyTorch 2.x safe helper functions

class EnsembleClassifier(nn.Module):
    BIRADS_CLASSES={i:f'BIRADS-{i}' for i in range(7)}
    def __init__(self,meta_dim=8,pretrained=False):
        super().__init__()
        # ViT-B/16 stream
        if TIMM_AVAILABLE:
            self.vit=timm.create_model('vit_base_patch16_224',pretrained=pretrained,
                                        num_classes=0,global_pool='token'); vit_dim=768
        else:
            b=_resnet50(pretrained)
            self.vit=nn.Sequential(*list(b.children())[:-1],nn.Flatten()); vit_dim=2048
        # DenseNet121 stream
        bd=_densenet121(pretrained)
        self.densenet=nn.Sequential(bd.features,nn.ReLU(True),nn.AdaptiveAvgPool2d(1),nn.Flatten())
        dense_dim=1024
        # EfficientNet-B3 stream
        if TIMM_AVAILABLE:
            self.effnet=timm.create_model('efficientnet_b3',pretrained=pretrained,
                                           num_classes=0,global_pool='avg'); eff_dim=1536
        else:
            be=_resnet50(pretrained)
            self.effnet=nn.Sequential(*list(be.children())[:-1],nn.Flatten()); eff_dim=2048
        total=vit_dim+dense_dim+eff_dim
        self.meta_enc=nn.Sequential(nn.Linear(meta_dim,64),nn.ReLU(),nn.Linear(64,128),nn.ReLU())
        self.fusion=nn.Sequential(nn.Linear(total+128,512),nn.GELU(),nn.Dropout(0.4),
                                   nn.Linear(512,256),nn.GELU(),nn.Dropout(0.3))
        self.head_birads=nn.Linear(256,7)
        self.head_cancer=nn.Linear(256,1)
        self.head_recall=nn.Linear(256,1)
    def forward(self,images,metadata):
        B,V,C,H,W=images.shape; flat=images.view(B*V,C,H,W)
        fv=self.vit(flat).view(B,V,-1).mean(1)
        fd=self.densenet(flat).view(B,V,-1).mean(1)
        fe=self.effnet(flat).view(B,V,-1).mean(1)
        h=self.fusion(torch.cat([fv,fd,fe,self.meta_enc(metadata)],dim=1))
        cp=torch.sigmoid(self.head_cancer(h)).squeeze(1)
        return {'birads_logits':self.head_birads(h),
                'birads_pred':torch.argmax(self.head_birads(h),dim=-1),
                'cancer_prob':cp,
                'recall_logit':self.head_recall(h).squeeze(1),
                'recall_pred':(cp>0.5).long()}

BIRADSScoringModel=EnsembleClassifier  # backward compat
birads_model=EnsembleClassifier(meta_dim=8,pretrained=False).to(DEVICE)
print(f'EnsembleClassifier (ViT-B/16 + DenseNet121 + EfficientNet-B3) params: {count_params(birads_model):,}')
with torch.no_grad():
    di=torch.randn(3,4,3,224,224).to(DEVICE); dm=torch.randn(3,8).to(DEVICE)
    bo=birads_model(di,dm)
print(f'  BIRADS: {bo["birads_pred"].tolist()}')
print(f'  Cancer probs: {bo["cancer_prob"].round(decimals=3).tolist()}')


## ⚙️ Section 7 — Training Pipeline & Operating Point Calibration

In [ ]:
# ─────────────────────────────────────────────────────────────────
# TRAINING PIPELINE WITH OPERATING POINT (OP) SELECTION
# AIMS study key finding: OP selection is site-specific
#   - Initial OP → recall rate 11-12% (too high)
#   - Recalibrated OP → 6.7-10% (closer to target)
# Target: AI recall rate ≤ first_reader + 2 percentage points
#         Predicted arbitration rate < 2.5× human-only rate
# ─────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    """Focal loss for class imbalance (1.7% cancer prevalence)."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets.float(), reduction='none')
        pt  = torch.exp(-bce)
        loss = self.alpha * (1 - pt) ** self.gamma * bce
        return loss.mean()

class AIMSTrainer:
    """
    Trainer with AIMS-specific operating point (OP) selection.
    Follows the iterative calibration process from the AIMS prospective study.
    """

    def __init__(self, model: nn.Module,
                 device: torch.device = DEVICE,
                 lr: float = 1e-4):
        self.model  = model
        self.device = device
        self.optim  = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        self.sched  = optim.lr_scheduler.CosineAnnealingLR(self.optim, T_max=50)
        self.focal  = FocalLoss(alpha=0.75, gamma=2.0)  # Upweight positives
        self.ce     = nn.CrossEntropyLoss()
        self.history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

    def compute_loss(self, out: dict, labels: dict) -> torch.Tensor:
        """Multi-task loss: cancer + BIRADS + recall."""
        cancer_lbl = labels['cancer'].float().to(self.device)
        birads_lbl = labels['birads_score'].to(self.device)

        # Clamp BIRADS to valid range
        birads_lbl = torch.clamp(birads_lbl, 0, 6)

        l_cancer  = self.focal(out['recall_logit'], cancer_lbl)
        l_birads  = self.ce(out['birads_logits'], birads_lbl)
        return l_cancer * 2.0 + l_birads * 0.5   # Weight cancer detection more

    def train_epoch(self, loader: DataLoader) -> float:
        self.model.train()
        total_loss = 0.0
        for imgs, labels in tqdm(loader, desc='Training', leave=False):
            imgs = imgs.to(self.device)
            meta = torch.randn(imgs.shape[0], 8).to(self.device)  # Mock metadata
            self.optim.zero_grad()
            out  = self.model(imgs, meta)
            loss = self.compute_loss(out, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.optim.step()
            total_loss += loss.item()
        self.sched.step()
        return total_loss / len(loader)

    @torch.no_grad()
    def validate(self, loader: DataLoader) -> dict:
        self.model.eval()
        all_probs, all_labels = [], []
        total_loss = 0.0
        for imgs, labels in loader:
            imgs = imgs.to(self.device)
            meta = torch.randn(imgs.shape[0], 8).to(self.device)
            out  = self.model(imgs, meta)
            loss = self.compute_loss(out, labels)
            total_loss += loss.item()
            all_probs.extend(out['cancer_prob'].cpu().numpy())
            all_labels.extend(labels['cancer'].numpy())
        # FIX: roc_auc_score crashes if only one class present in batch
        auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5
        return {'val_loss': total_loss/len(loader), 'val_auc': auc,
                'probs': np.array(all_probs), 'labels': np.array(all_labels)}

    def select_operating_point(self, probs: np.ndarray,
                                labels: np.ndarray,
                                target_recall_rate: float = 0.065,
                                human_arb_rate: float = 0.055) -> dict:
        """
        AIMS OP selection strategy:
        Maximize sensitivity while keeping:
          - Recall rate ≤ first_reader + 2 pp
          - Predicted arbitration rate < 2.5× human-only
        Returns: optimal threshold and metrics at that threshold.
        """
        thresholds = np.linspace(0.01, 0.99, 500)
        results    = []

        for t in thresholds:
            pred = (probs >= t).astype(int)
            tp   = ((pred==1) & (labels==1)).sum()
            fp   = ((pred==1) & (labels==0)).sum()
            fn   = ((pred==0) & (labels==1)).sum()
            tn   = ((pred==0) & (labels==0)).sum()
            n    = len(labels)

            sensitivity  = tp / (tp+fn+1e-8)
            specificity  = tn / (tn+fp+1e-8)
            recall_rate  = (tp+fp) / n
            # Simplistic arbitration rate estimate (disagreement)
            arb_rate_est = recall_rate * 0.6  # ~60% go to arbitration
            cdr          = tp / (n/1000)  # per 1000

            # AIMS constraints
            within_recall  = recall_rate <= (target_recall_rate + 0.02)
            within_arb     = arb_rate_est < (human_arb_rate * 2.5)

            results.append({
                'threshold'  : t,
                'sensitivity': sensitivity,
                'specificity': specificity,
                'recall_rate': recall_rate,
                'cdr_per_1k' : cdr,
                'arb_rate'   : arb_rate_est,
                'eligible'   : within_recall and within_arb,
                'youden'     : sensitivity + specificity - 1,
            })

        df_r = pd.DataFrame(results)
        eligible = df_r[df_r.eligible]
        if eligible.empty:
            best = df_r.loc[df_r.youden.idxmax()]
        else:
            best = eligible.loc[eligible.sensitivity.idxmax()]

        return best.to_dict()

# ─── Simulate training with mock dataset ─────────────────────────
# FIX: df_all guard - prevents NameError if cells run out of order
if 'df_all' not in dir():
    print('WARNING: df_all not found - recreating'); df_all=simulate_aims_dataframe(n=2000)
    df_train=df_all[df_all.split=='train'].copy(); df_val=df_all[df_all.split=='val'].copy()
    df_test=df_all[df_all.split=='test'].copy()
df_train = df_all[df_all.split=='train'].copy()
df_val   = df_all[df_all.split=='val'].copy()

if "df_all" not in dir():
    print("WARNING: recreating df_all"); df_all=simulate_aims_dataframe(); df_train=df_all[df_all.split=="train"].copy(); df_val=df_all[df_all.split=="val"].copy(); df_test=df_all[df_all.split=="test"].copy()
if not ("df_all" in dir()):
    print("WARNING: recreating df_all"); df_all=simulate_aims_dataframe(); df_train=df_all[df_all.split=="train"].copy(); df_val=df_all[df_all.split=="val"].copy()
train_ds = MammographyDataset(df_train, augment=True,  mode='cancer')
val_ds   = MammographyDataset(df_val,   augment=False, mode='cancer')

# Class-balanced sampler (1.7% cancer → heavily imbalanced)
labels_arr   = df_train.cancer_label.values.astype(int)  # FIX: explicit int for bincount
class_counts = np.bincount(labels_arr)
weights      = 1.0 / class_counts[labels_arr]
sampler      = WeightedRandomSampler(weights, len(weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=8, sampler=sampler, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0)

trainer = AIMSTrainer(birads_model, DEVICE)

print('🏋️  Training for 2 demo epochs...')
for epoch in range(2):
    train_loss = trainer.train_epoch(train_loader)
    val_metrics = trainer.validate(val_loader)
    print(f'  Epoch {epoch+1}: train_loss={train_loss:.4f} | '
          f'val_loss={val_metrics["val_loss"]:.4f} | '
          f'val_AUC={val_metrics["val_auc"]:.3f}')

# OP selection
op = trainer.select_operating_point(
    val_metrics['probs'], val_metrics['labels'],
    target_recall_rate=0.065
)
print(f'\n⚙️  Selected Operating Point:')
print(f'   Threshold  : {op["threshold"]:.3f}')
print(f'   Sensitivity: {op["sensitivity"]:.3f}')
print(f'   Specificity: {op["specificity"]:.3f}')
print(f'   Recall Rate: {op["recall_rate"]*100:.1f}%')
print(f'   CDR/1000   : {op["cdr_per_1k"]:.2f}')

## 🎯 Section 8 — Cancer Risk Score & Tumor Identification

In [ ]:
# ─────────────────────────────────────────────────────────────────
# CANCER RISK SCORE COMPUTATION & VISUALIZATION
# Benchmarked against AIMS study published metrics:
#   - Overall sensitivity: 0.541 vs 0.437 (first reader)
#   - Specificity: 0.943 (noninferior to 0.952)
#   - AUC = 0.83 (case), 0.84 (breast), 0.978 (screen-detected only)
#   - Invasive cancer sensitivity: 0.54 vs 0.43 (first reader)
# ─────────────────────────────────────────────────────────────────

class RiskScoreEvaluator:
    """
    Comprehensive risk evaluation matching AIMS study endpoints:
    - Primary: noninferiority (5% margin) vs first reader
    - Secondary: vs second reader, consensus, breast-level
    - CDR, recall rate, PPV, NPV, subgroup analyses
    """

    NI_MARGIN = 0.05  # 5% noninferiority margin (AIMS prespecified)

    def evaluate_at_threshold(self, probs: np.ndarray,
                               labels: np.ndarray,
                               threshold: float,
                               n_total: int = None) -> dict:
        """Full metrics at a given OP threshold."""
        n_total = n_total or len(labels)
        pred = (probs >= threshold).astype(int)

        tp = int(((pred==1) & (labels==1)).sum())
        fp = int(((pred==1) & (labels==0)).sum())
        fn = int(((pred==0) & (labels==1)).sum())
        tn = int(((pred==0) & (labels==0)).sum())

        sensitivity = tp / (tp + fn + 1e-8)
        specificity = tn / (tn + fp + 1e-8)
        ppv = tp / (tp + fp + 1e-8)
        npv = tn / (tn + fn + 1e-8)
        recall_rate = (tp + fp) / n_total
        cdr_per_1k  = tp / (n_total / 1000)

        try:
            auc = roc_auc_score(labels, probs)
        except Exception:
            auc = 0.5

        return {
            'sensitivity': sensitivity, 'specificity': specificity,
            'ppv': ppv, 'npv': npv,
            'recall_rate': recall_rate, 'cdr_per_1k': cdr_per_1k,
            'auc': auc, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'threshold': threshold
        }

    def noninferiority_test(self, ai_sensitivity: float,
                             human_sensitivity: float,
                             n_pos: int) -> dict:
        """
        Wald noninferiority test (AIMS primary endpoint).
        H0: AI sensitivity < human sensitivity - NI_MARGIN
        """
        diff = ai_sensitivity - human_sensitivity
        se   = np.sqrt(
            ai_sensitivity*(1-ai_sensitivity)/n_pos +
            human_sensitivity*(1-human_sensitivity)/n_pos
        )
        z_stat = (diff + self.NI_MARGIN) / (se + 1e-8)
        p_val  = 1 - stats.norm.cdf(z_stat)
        ci_low = diff - 1.96*se
        ci_up  = diff + 1.96*se
        noninferior = p_val < 0.025  # One-sided at 0.025 level
        return {
            'difference': diff, 'se': se, 'z_stat': z_stat,
            'p_value': p_val, 'ci': (ci_low, ci_up),
            'noninferior': noninferior,
            'superior': diff > 0 and p_val < 0.025
        }

    def plot_roc_aims_style(self, probs: np.ndarray,
                             labels: np.ndarray,
                             human_sens: float = 0.437,
                             human_spec: float = 0.952,
                             ax=None, title: str = 'ROC Curve'):
        """Plot ROC with AIMS-style human reader benchmark points."""
        if ax is None:
            fig, ax = plt.subplots(1, 1, figsize=(7, 7))

        fpr, tpr, _ = roc_curve(labels, probs)
        auc = roc_auc_score(labels, probs)

        ax.plot(fpr, tpr, color='#FF6B00', lw=2.5,
                label=f'AI Model [AUC = {auc:.3f}]')
        ax.plot([0,1],[0,1], 'k--', alpha=0.4, label='Random')

        # AIMS benchmark points
        ax.scatter(1-human_spec, human_sens, s=180, marker='o',
                   color='royalblue', zorder=5, label='1st Human Reader')
        ax.scatter(1-0.934, 0.500, s=180, marker='D',
                   color='green', zorder=5, label='2nd Human Reader')
        ax.scatter(1-0.945, 0.478, s=180, marker='s',
                   color='purple', zorder=5, label='Human Consensus')

        ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
        ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='lower right', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0,1]); ax.set_ylim([0,1])

        # Annotate target region (AIMS AI operating point zone)
        ax.axhspan(0.50, 0.60, alpha=0.05, color='orange',
                   label='AIMS AI OP zone')
        return ax

    def subgroup_analysis(self, df: pd.DataFrame,
                           probs: np.ndarray,
                           threshold: float,
                           subgroup_col: str) -> pd.DataFrame:
        """Compute sensitivity/specificity by subgroup (AIMS fairness analysis)."""
        df = df.copy()
        df['prob']  = probs
        df['pred']  = (probs >= threshold).astype(int)
        results = []

        for grp, gdf in df.groupby(subgroup_col):
            m = self.evaluate_at_threshold(
                gdf['prob'].values, gdf['cancer_label'].values,
                threshold, len(gdf)
            )
            m['subgroup'] = grp
            m['n_cases']  = len(gdf)
            m['n_pos']    = int(gdf['cancer_label'].sum())
            results.append(m)

        return pd.DataFrame(results).sort_values('subgroup')

# ─── Evaluate with simulated probabilities ──────────────────────
evaluator = RiskScoreEvaluator()
df_test   = df_all[df_all.split=='test'].copy()

# Generate realistic probabilities from risk_score column
probs_sim  = df_test.risk_score.values
labels_sim = df_test.cancer_label.values

threshold = op['threshold']
metrics   = evaluator.evaluate_at_threshold(probs_sim, labels_sim, threshold)

print('📊 Model Performance vs AIMS Study Benchmarks')
print('='*55)
print(f"{'Metric':<25} {'Our Model':>12} {'AIMS AI':>10} {'AIMS R1':>10}")
print('-'*55)
print(f"{'Sensitivity':<25} {metrics['sensitivity']:>12.3f} {'0.541':>10} {'0.437':>10}")
print(f"{'Specificity':<25} {metrics['specificity']:>12.3f} {'0.943':>10} {'0.952':>10}")
print(f"{'Recall Rate':<25} {metrics['recall_rate']:>12.3f} {'0.065':>10} {'0.055':>10}")
print(f"{'CDR per 1,000':<25} {metrics['cdr_per_1k']:>12.2f} {'9.33':>10} {'7.54':>10}")
print(f"{'AUC':<25} {metrics['auc']:>12.3f} {'0.830':>10} {'   -':>10}")
print('='*55)

# NI test
ni = evaluator.noninferiority_test(metrics['sensitivity'], 0.437, metrics['tp'])
print(f"\n🧪 Noninferiority Test vs R1 (5% margin):")
print(f"   Difference = {ni['difference']:+.3f} (95% CI: {ni['ci'][0]:.3f}, {ni['ci'][1]:.3f})")
print(f"   p = {ni['p_value']:.4f} → {'✅ NONINFERIOR' if ni['noninferior'] else '❌ NOT noninferior'}")
if ni['superior']:
    print(f"   🏆 SUPERIOR (p={ni['p_value']:.4f})")

In [ ]:
# ─── Full Visualization Panel ────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('AIMS-Inspired Breast Cancer AI — Performance Dashboard',
             fontsize=16, fontweight='bold', y=1.01)

# 1. ROC Curve
evaluator.plot_roc_aims_style(probs_sim, labels_sim, ax=axes[0,0],
                               title='(a) ROC Curve — Case Level')

# 2. BIRADS Density Distribution
ax2 = axes[0,1]
birads_counts = df_all.density_birads.value_counts().sort_index()
colors_b = ['#2ECC71','#3498DB','#E67E22','#E74C3C']
bars = ax2.bar([f'B{i}' for i in birads_counts.index],
                birads_counts.values, color=colors_b)
ax2.set_title('(b) BIRADS Density Distribution', fontweight='bold')
ax2.set_xlabel('Density Category'); ax2.set_ylabel('Count')
for bar, v in zip(bars, birads_counts.values):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
             str(v), ha='center', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# 3. Cancer time-point breakdown
ax3 = axes[0,2]
tp_map   = {0:'Negative',1:'Screen\nDetected',2:'Interval',3:'Next\nRound'}
tp_counts = df_all.cancer_timepoint.value_counts().sort_index()
colors_tp = ['#95A5A6','#27AE60','#E74C3C','#9B59B6']
ax3.pie(tp_counts.values, labels=[tp_map[i] for i in tp_counts.index],
         colors=colors_tp, autopct='%1.1f%%', startangle=90)
ax3.set_title('(c) Cancer Time-Point Distribution', fontweight='bold')

# 4. Subgroup sensitivity analysis (ethnicity)
ax4 = axes[1,0]
sg_eth = evaluator.subgroup_analysis(df_test, probs_sim, threshold, 'ethnicity')
sg_eth_f = sg_eth[sg_eth.n_pos > 2]  # Filter small groups
bars4 = ax4.barh(sg_eth_f['subgroup'], sg_eth_f['sensitivity'],
                  color='steelblue', alpha=0.8)
ax4.axvline(x=metrics['sensitivity'], color='red', linestyle='--',
             linewidth=2, label='Overall AI')
ax4.axvline(x=0.437, color='blue', linestyle=':', linewidth=2, label='AIMS R1')
ax4.axvline(x=metrics['sensitivity']-0.05, color='orange', linestyle=':',
             linewidth=1.5, label='NI margin')
ax4.set_title('(d) Sensitivity by Ethnicity (Fairness)', fontweight='bold')
ax4.set_xlabel('Sensitivity'); ax4.legend(fontsize=8)
ax4.grid(True, alpha=0.3, axis='x')

# 5. Subgroup sensitivity by age group
ax5 = axes[1,1]
df_test2 = df_test.copy()
df_test2['age_group'] = pd.cut(df_test2.age, bins=[49,54,59,64,70],
                                labels=['50-54','55-59','60-64','65-70'])
probs_test2 = probs_sim  # same size
sg_age = evaluator.subgroup_analysis(df_test2, probs_test2, threshold, 'age_group')
sg_age_f = sg_age[sg_age.n_pos > 2]
ax5.bar(sg_age_f['subgroup'].astype(str), sg_age_f['sensitivity'],
         color='teal', alpha=0.8)
ax5.axhline(y=metrics['sensitivity'], color='red', linestyle='--', label='Overall')
ax5.set_title('(e) Sensitivity by Age Group', fontweight='bold')
ax5.set_xlabel('Age Group'); ax5.set_ylabel('Sensitivity')
ax5.legend(); ax5.grid(True, alpha=0.3, axis='y')

# 6. CDR vs Recall Rate (AIMS-style trade-off)
ax6 = axes[1,2]
thresholds_range = np.linspace(0.1, 0.9, 50)
cdrs, recalls = [], []
for t in thresholds_range:
    m = evaluator.evaluate_at_threshold(probs_sim, labels_sim, t)
    cdrs.append(m['cdr_per_1k'])
    recalls.append(m['recall_rate'] * 100)

ax6.plot(recalls, cdrs, color='#FF6B00', lw=2.5, label='AI (this model)')
ax6.scatter([5.5], [9.33], s=200, marker='s', color='#FF6B00',
             zorder=5, label='AIMS AI OP')
ax6.scatter([5.5], [7.54], s=200, marker='o', color='royalblue',
             zorder=5, label='AIMS R1')
ax6.scatter([5.5], [8.03], s=200, marker='D', color='green',
             zorder=5, label='AIMS Consensus')
ax6.set_xlabel('Recall Rate (%)', fontsize=11)
ax6.set_ylabel('Cancer Detection Rate (per 1,000)', fontsize=11)
ax6.set_title('(f) CDR vs Recall Rate Trade-off', fontweight='bold')
ax6.legend(fontsize=9); ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/aims_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Performance dashboard saved')

## 🔮 Section 9 — Interval Cancer & Future Cancer Prediction

In [ ]:
# ─────────────────────────────────────────────────────────────────
# INTERVAL CANCER & NEXT-ROUND CANCER PREDICTION
# AIMS KEY FINDING:
#   - 25.0% of INTERVAL cancers correctly identified at screening
#   - 25.1% of NEXT-ROUND cancers correctly identified
#   - 88.0% localized to correct breast
#   - 58.1% localized to precise lesion
# Warren et al. (companion paper): AI arm 32.4% IC sens before arb
#   but drops to 8.8% after arbitration (readers overrule AI)
# ─────────────────────────────────────────────────────────────────

class IntervalCancerPredictor:
    """
    Predicts future cancer risk (interval + next-round).
    Uses: current screening features + temporal risk modeling.
    Models the 3-year follow-up window used in AIMS.
    """

    # AIMS sensitivity benchmarks for validation
    AIMS_BENCHMARKS = {
        'screen_detected' : {'ai': 0.541, 'r1': 0.437, 'r2': 0.500},
        'interval_cancer' : {'ai': 0.250, 'r1': 0.000, 'r2': 0.000},
        'next_round'      : {'ai': 0.251, 'r1': 0.000, 'r2': 0.000},
        # Warren companion: BEFORE arbitration
        'interval_pre_arb': {'ai': 0.324, 'r1': 0.154, 'r2': None},
        'next_rnd_pre_arb': {'ai': 0.340, 'r1': 0.128, 'r2': None},
        # AFTER arbitration (AI arm)
        'interval_post_arb': {'ai': 0.088, 'human': 0.059},
        'next_rnd_post_arb': {'ai': 0.081, 'human': 0.055},
    }

    def __init__(self, base_model: nn.Module):
        self.model = base_model
        # Temporal risk head (added to base model features)
        self.temporal_head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(128, 4),  # [neg, screen_detected, interval, next_round]
        ).to(DEVICE)
        self.optim = optim.Adam(self.temporal_head.parameters(), lr=5e-5)

    def predict_timepoint(self, cancer_prob: float,
                           density_birads: int,
                           age: int,
                           screen_type: str,
                           roi_confidence: float = 0.0) -> dict:
        """
        Heuristic time-point risk assignment:
        Mirrors AIMS finding that AI captures 25% of future cancers.
        """
        # Risk factors from AIMS study
        density_factor = {1: 0.8, 2: 1.0, 3: 1.3, 4: 1.6}.get(density_birads, 1.0)
        age_factor     = 1.0 + max(0, (age - 60)) * 0.02
        prev_factor    = 1.5 if screen_type == 'prevalent' else 1.0

        base_risk = cancer_prob * density_factor * age_factor

        # Estimate time-point probabilities
        p_screen    = min(0.95, base_risk * 0.65)
        p_interval  = min(0.30, base_risk * 0.25 * (1 - roi_confidence))
        p_next_round = min(0.30, base_risk * 0.25 * prev_factor)
        p_negative  = max(0.0, 1 - p_screen - p_interval - p_next_round)

        # Normalize
        total = p_negative + p_screen + p_interval + p_next_round
        probs = {
            'negative'      : p_negative / total,
            'screen_detected': p_screen / total,
            'interval'      : p_interval / total,
            'next_round'    : p_next_round / total,
        }
        predicted_tp = max(probs, key=probs.get)

        # 3-year follow-up window risk score (AIMS: 39-month ground truth)
        future_risk_39m = p_interval + p_next_round

        return {
            'timepoint_probs'  : probs,
            'predicted_timepoint': predicted_tp,
            'future_risk_39m'  : future_risk_39m,
            'recommend_early_recall':
                future_risk_39m > 0.15,  # Trigger earlier follow-up
            'localization_recommendation':
                'Lesion identified' if roi_confidence > 0.5
                else 'Subtle/diffuse — bilateral comparison needed'
        }

    def evaluate_by_timepoint(self, df: pd.DataFrame,
                               probs: np.ndarray,
                               threshold: float) -> pd.DataFrame:
        """Evaluate sensitivity separately per cancer time-point."""
        df = df.copy()
        df['prob'] = probs
        df['pred'] = (probs >= threshold).astype(int)

        rows = []
        for tp_code, tp_name in {
            0: 'Negative',
            1: 'Screen Detected',
            2: 'Interval Cancer',
            3: 'Next Round'
        }.items():
            subset = df[df.cancer_timepoint == tp_code]
            if len(subset) == 0:
                continue
            if tp_code == 0:
                spec = (subset.pred == 0).mean()
                rows.append({'timepoint': tp_name, 'n': len(subset),
                              'specificity': spec, 'sensitivity': None})
            else:
                sens = (subset.pred == 1).mean() if len(subset) > 0 else 0
                aims_bench = self.AIMS_BENCHMARKS.get(
                    tp_name.lower().replace(' ','_'), {})
                rows.append({
                    'timepoint'   : tp_name,
                    'n'           : len(subset),
                    'sensitivity' : sens,
                    'aims_ai'     : aims_bench.get('ai', '-'),
                    'aims_r1'     : aims_bench.get('r1', '-'),
                    'specificity' : None
                })
        return pd.DataFrame(rows)

ic_predictor = IntervalCancerPredictor(birads_model)

# Demo prediction
sample_pred = ic_predictor.predict_timepoint(
    cancer_prob=0.45, density_birads=3, age=58,
    screen_type='incident', roi_confidence=0.3
)
print('🔮 Future Cancer Prediction:')
print(f"   Time-point probs : {' | '.join([f'{k}: {v:.2%}' for k,v in sample_pred['timepoint_probs'].items()])}")
print(f"   Predicted         : {sample_pred['predicted_timepoint']}")
print(f"   39-month risk     : {sample_pred['future_risk_39m']:.2%}")
print(f"   Early recall?     : {sample_pred['recommend_early_recall']}")
print(f"   Note              : {sample_pred['localization_recommendation']}")

# Timepoint analysis
tp_results = ic_predictor.evaluate_by_timepoint(df_test, probs_sim, threshold)
print('\n📊 Sensitivity/Specificity by Cancer Time-Point:')
print(tp_results.to_string(index=False))

## ⚖️ Section 10 — Fairness & Subgroup Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────
# FAIRNESS ANALYSIS — AIMS KEY FINDING:
#   "No systematic demographic disparities were observed"
#   Borderline failures (5% NI margin):
#     - IMD decile 1 (most deprived, n=2,192, only 43 pos)
#     - Mixed ethnicity (n=1,132, only 21 pos)
#   AIMS recommendation: ~150-200 screen-detected cancers per
#   subgroup needed for reliable inference
# ─────────────────────────────────────────────────────────────────

class FairnessAnalyzer:
    """Comprehensive fairness analysis following AIMS study methodology."""

    SUBGROUPS = {
        'By Ethnicity'     : 'ethnicity',
        'By Age Group'     : 'age_group',
        'By IMD Decile'    : 'imd_decile',
        'By Manufacturer'  : 'manufacturer',
        'By Screen Type'   : 'screen_type',
        'By Density BIRADS': 'density_birads',
    }

    def __init__(self, ni_margin: float = 0.05):
        self.ni_margin = ni_margin

    def run_all_subgroups(self, df: pd.DataFrame,
                           probs: np.ndarray,
                           threshold: float) -> Dict[str, pd.DataFrame]:
        """Run subgroup analysis for all AIMS-defined subgroups."""
        df = df.copy()
        df['prob'] = probs
        df['pred'] = (probs >= threshold).astype(int)
        # Add age groups
        df['age_group'] = pd.cut(df.age, bins=[49,54,59,64,70],
                                  labels=['50-54','55-59','60-64','65+'])

        results = {}
        for name, col in self.SUBGROUPS.items():
            if col not in df.columns:
                continue
            rows = []
            for grp, gdf in df.groupby(col, observed=True):
                n_pos = int(gdf.cancer_label.sum())
                n_neg = len(gdf) - n_pos

                if n_pos >= 5:   # minimum for sensitivity
                    sens = (gdf[gdf.cancer_label==1].pred == 1).mean()
                else:
                    sens = np.nan

                if n_neg >= 20:
                    spec = (gdf[gdf.cancer_label==0].pred == 0).mean()
                else:
                    spec = np.nan

                # Calibration: does prob track cancer rate?
                obs_rate  = gdf.cancer_label.mean()
                mean_pred = gdf.prob.mean()
                calib_err = abs(mean_pred - obs_rate)

                # Check NI against overall sensitivity
                overall_sens = df[df.cancer_label==1].pred.mean()
                ni_pass = (not np.isnan(sens)) and \
                           (sens >= overall_sens - self.ni_margin)

                rows.append({
                    'subgroup'    : str(grp),
                    'n_cases'     : len(gdf),
                    'n_pos'       : n_pos,
                    'sensitivity' : sens,
                    'specificity' : spec,
                    'recall_rate' : gdf.pred.mean(),
                    'calib_error' : calib_err,
                    'ni_pass'     : ni_pass,
                    'flag'        : '⚠️' if not ni_pass and not np.isnan(sens) else '✅'
                })
            results[name] = pd.DataFrame(rows)
        return results

    def plot_fairness_grid(self, results: dict):
        """Plot fairness forest plot (AIMS Figure 3 style)."""
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.suptitle('Fairness Analysis — Sensitivity by Subgroup\n'
                     '(Gray band = 5% Noninferiority Margin, AIMS Study Methodology)',
                     fontsize=14, fontweight='bold')
        axes_flat = axes.flatten()

        for ax, (name, df_sg) in zip(axes_flat, results.items()):
            df_sg = df_sg.dropna(subset=['sensitivity'])
            if df_sg.empty:
                ax.set_visible(False); continue

            overall = df_sg.sensitivity.mean()
            colors  = ['#E74C3C' if not r else '#27AE60'
                       for r in df_sg.ni_pass]

            ax.barh(df_sg.subgroup.astype(str), df_sg.sensitivity,
                    color=colors, alpha=0.8)
            ax.axvline(overall, color='black', linestyle='--',
                       linewidth=2, label=f'Overall ({overall:.3f})')
            ax.axvspan(overall - 0.05, overall,
                       alpha=0.1, color='gray', label='NI margin')
            ax.set_title(name, fontweight='bold')
            ax.set_xlabel('Sensitivity')
            ax.legend(fontsize=8)
            ax.grid(True, alpha=0.3, axis='x')
            ax.set_xlim([0, 1])

            # Annotate n
            for i, (_, row) in enumerate(df_sg.iterrows()):
                ax.text(0.02, i, f'n={row.n_pos}', va='center',
                        fontsize=7, color='white',
                        fontweight='bold')

        plt.tight_layout()
        plt.savefig('/tmp/fairness_grid.png', dpi=150, bbox_inches='tight')
        plt.show()

# ─── Run fairness analysis ───────────────────────────────────────
fa = FairnessAnalyzer(ni_margin=0.05)
fairness_results = fa.run_all_subgroups(df_test, probs_sim, threshold)

print('⚖️  Fairness Summary (AIMS-style):')
for subgrp_name, df_sg in fairness_results.items():
    fails = df_sg[~df_sg.ni_pass & df_sg.sensitivity.notna()]
    status = f'{len(fails)} subgroup(s) below NI margin' if len(fails) > 0 else 'All passed'
    print(f'   {subgrp_name}: {status}')
    if len(fails) > 0:
        print(df_sg[['subgroup','n_pos','sensitivity','flag']]
              .to_string(index=False))

fa.plot_fairness_grid(fairness_results)

## 📡 Section 11 — Distribution Shift Monitor & Drift Detection

In [ ]:
# ─────────────────────────────────────────────────────────────────
# DISTRIBUTION SHIFT MONITORING
# AIMS KEY FINDING: 8-year lag between tuning (2016) and deployment
#   caused significant drift:
#   - Initial recall rate: 11.3% & 12.3% (target: ~6%)
#   - Hologic Selenia Dimensions: 10.9% vs 6.3% (Lorad Selenia)
#   - Human readers became more specific over time
# Monitor: recall rate proxy, week-over-week variation
# Trigger: recalibrate OP when recall rate drifts > 2 pp from target
# ─────────────────────────────────────────────────────────────────

class DistributionShiftMonitor:
    """
    Continuous monitoring system for deployment drift.
    Implements the iterative OP recalibration from AIMS prospective study.
    """

    def __init__(self, target_recall_rate: float = 0.065,
                 window_size: int = 500,
                 drift_threshold_pp: float = 2.0):
        self.target_rr       = target_recall_rate
        self.window_size     = window_size
        self.drift_threshold = drift_threshold_pp / 100.0
        self.predictions     = []   # Rolling window
        self.weekly_stats    = []
        self.op_history      = []

    def add_batch(self, probs: np.ndarray,
                  threshold: float,
                  week: int = 0,
                  labels: Optional[np.ndarray] = None):
        """Add new batch of predictions to the rolling window."""
        preds = (probs >= threshold).astype(int)
        recall_rate = preds.mean()

        entry = {
            'week'        : week,
            'n_cases'     : len(probs),
            'recall_rate' : recall_rate,
            'mean_prob'   : probs.mean(),
            'threshold'   : threshold,
            'drift_flag'  : abs(recall_rate - self.target_rr) > self.drift_threshold,
        }
        if labels is not None and len(set(labels)) > 1:
            try: entry['auc']=roc_auc_score(labels,probs)  # FIX: guard single-class batch
            except ValueError: entry['auc']=None
        self.weekly_stats.append(entry)
        # Rolling window
        self.predictions.extend(probs.tolist())
        if len(self.predictions) > self.window_size:
            self.predictions = self.predictions[-self.window_size:]

    def check_drift(self) -> dict:
        """Statistical drift test using KS test on prediction distribution."""
        if len(self.weekly_stats) < 3:
            return {'status': 'INSUFFICIENT_DATA'}
        df_w = pd.DataFrame(self.weekly_stats)
        recent = df_w.tail(4).recall_rate.mean()
        baseline = df_w.head(min(4, len(df_w))).recall_rate.mean()
        drift_pp = (recent - baseline) * 100
        drift_from_target = (recent - self.target_rr) * 100

        status = 'OK'
        if abs(drift_from_target) > 2.0:
            status = 'RECALIBRATION_NEEDED'
        elif abs(drift_pp) > 1.5:
            status = 'WATCH'

        return {
            'status'           : status,
            'current_rr'       : recent,
            'baseline_rr'      : baseline,
            'target_rr'        : self.target_rr,
            'drift_pp'         : drift_pp,
            'drift_from_target': drift_from_target,
            'weeks_monitored'  : len(df_w),
            'action_required'  : status == 'RECALIBRATION_NEEDED'
        }

    def suggest_new_threshold(self, probs_recent: np.ndarray,
                               target_rr: float) -> float:
        """Binary search for threshold that achieves target recall rate."""
        lo, hi = 0.01, 0.99
        for _ in range(50):
            mid = (lo + hi) / 2
            rr  = (probs_recent >= mid).mean()
            if rr > target_rr:
                lo = mid
            else:
                hi = mid
        return (lo + hi) / 2

    def plot_weekly_monitoring(self):
        """AIMS-style weekly monitoring plot."""
        if not self.weekly_stats:
            print('No data yet'); return
        df_w = pd.DataFrame(self.weekly_stats)
        fig, axes = plt.subplots(2, 1, figsize=(14, 8))
        fig.suptitle('Weekly AI Monitoring — AIMS Prospective Deployment Style',
                     fontsize=13, fontweight='bold')

        # Recall rate
        ax1 = axes[0]
        ax1.plot(df_w.week, df_w.recall_rate*100, 'o-',
                  color='#E74C3C', lw=2, markersize=8, label='AI Recall Rate')
        ax1.axhline(self.target_rr*100, color='green', linestyle='--',
                     lw=2, label=f'Target ({self.target_rr*100:.1f}%)')
        ax1.axhline((self.target_rr+0.02)*100, color='orange', linestyle=':',
                     lw=1.5, label='Max tolerance (+2pp)')
        ax1.fill_between(df_w.week,
                          (self.target_rr-0.01)*100,
                          (self.target_rr+0.02)*100,
                          alpha=0.15, color='green', label='Acceptable zone')
        # Flag drift weeks
        drift_weeks = df_w[df_w.drift_flag]
        if not drift_weeks.empty:
            ax1.scatter(drift_weeks.week, drift_weeks.recall_rate*100,
                         s=200, marker='X', color='red', zorder=5, label='Drift alert')
        ax1.set_ylabel('Recall Rate (%)', fontsize=11)
        ax1.set_title('Weekly Recall Rate Monitoring')
        ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

        # Cases per week
        ax2 = axes[1]
        ax2.bar(df_w.week, df_w.n_cases, color='steelblue', alpha=0.7)
        ax2.set_ylabel('Cases per Week', fontsize=11)
        ax2.set_xlabel('Study Week', fontsize=11)
        ax2.set_title('Weekly Case Volume (cohort variation)')
        ax2.grid(True, alpha=0.3, axis='y')

        plt.tight_layout()
        plt.savefig('/tmp/monitoring.png', dpi=150, bbox_inches='tight')
        plt.show()

# ─── Simulate AIMS prospective deployment (8 weeks) ──────────────
rng   = np.random.default_rng(42)
monitor = DistributionShiftMonitor(target_recall_rate=0.065)

print('📡 Simulating AIMS prospective deployment (8 weeks)...')
initial_threshold  = 0.35  # Too low → too many recalls (AIMS initial OP)
adjusted_threshold = 0.55  # Recalibrated after 2 weeks

for week in range(1, 9):
    n_cases  = rng.integers(300, 900)
    # Simulate distribution shift: early weeks more sensitive (lower scores needed)
    base_mu  = 0.12 if week <= 2 else 0.08   # Drift in score distribution
    probs_w  = np.clip(rng.normal(base_mu, 0.15, n_cases), 0, 1)
    t        = initial_threshold if week <= 2 else adjusted_threshold
    monitor.add_batch(probs_w, threshold=t, week=week)

    drift = monitor.check_drift()
    flag  = '🔴 DRIFT' if drift.get('action_required') else ('🟡 WATCH' if drift['status']=='WATCH' else '🟢 OK')
    print(f'   Week {week:2d}: n={n_cases:4d} | '
          f'RR={monitor.weekly_stats[-1]["recall_rate"]*100:.1f}% | '
          f'status={flag}')

monitor.plot_weekly_monitoring()
print(f'\n📊 Final drift assessment:')
final = monitor.check_drift()
for k, v in final.items():
    if isinstance(v, float): print(f'   {k}: {v:.4f}')
    else: print(f'   {k}: {v}')

## 🚀 Section 12 — Full End-to-End Inference Pipeline & Report

In [ ]:
# GRAD-CAM EXPLAINABILITY
# NEW: Was entirely missing despite being listed in notebook title
# Uses pytorch-grad-cam library; manual implementation as fallback

class MammographyGradCAM:
    def __init__(self,model):
        self.model=model.eval(); self._grads=None; self._acts=None
        self.target_layer=self._find_last_conv(model)
    def _find_last_conv(self,m):
        last=None
        for mod in m.modules():
            if isinstance(mod,nn.Conv2d): last=mod
        return last
    def generate_heatmap(self,img_t,target_class=1):
        if GRADCAM_AVAILABLE and self.target_layer is not None:
            return self._library_gradcam(img_t,target_class)
        return self._manual_gradcam(img_t,target_class)
    def _library_gradcam(self,img_t,target_class):
        class Wrap(nn.Module):
            def __init__(s,m): super().__init__(); s.m=m
            def forward(s,x):
                meta=torch.zeros(x.shape[0],8).to(x.device)
                imgs=x.unsqueeze(1).expand(-1,4,-1,-1,-1)
                out=s.m(imgs,meta)
                return torch.stack([1-out['cancer_prob'],out['cancer_prob']],dim=1)
        try:
            with GradCAM(model=Wrap(self.model),target_layers=[self.target_layer]) as cam:
                return cam(img_t,[ClassifierOutputTarget(target_class)])[0]
        except Exception as e:
            return self._manual_gradcam(img_t,target_class)
    def _manual_gradcam(self,img_t,target_class):
        if self.target_layer is None:
            return np.zeros(img_t.shape[-2:],dtype=np.float32)
        grads=[None]; acts=[None]
        fh=self.target_layer.register_forward_hook(lambda m,i,o: acts.__setitem__(0,o.detach()))
        bh=self.target_layer.register_full_backward_hook(lambda m,gi,go: grads.__setitem__(0,go[0].detach()))
        self.model.zero_grad()
        img=img_t.clone().requires_grad_(True)
        meta=torch.zeros(1,8).to(img.device)
        imgs4=img.unsqueeze(1).expand(-1,4,-1,-1,-1)
        out=self.model(imgs4,meta)
        score=out['cancer_prob'][0] if target_class==1 else (1-out['cancer_prob'][0])
        score.backward(); fh.remove(); bh.remove()
        if grads[0] is None or acts[0] is None:
            return np.zeros(img_t.shape[-2:],dtype=np.float32)
        w=grads[0].mean(dim=(2,3),keepdim=True)
        cam=(w*acts[0]).sum(1).squeeze()
        cam=F.relu(cam).cpu().numpy()
        if cam.max()>0: cam=cam/cam.max()
        H,W=img_t.shape[-2:]
        return cv2.resize(cam,(W,H)).astype(np.float32)
    def overlay(self,img_np,heatmap,alpha=0.5):
        hm=cv2.applyColorMap((heatmap*255).astype(np.uint8),cv2.COLORMAP_JET)
        hm=cv2.cvtColor(hm,cv2.COLOR_BGR2RGB)
        return (alpha*hm+(1-alpha)*img_np).astype(np.uint8)

grad_cam=MammographyGradCAM(birads_model)
print(f'Grad-CAM: {"pytorch-grad-cam" if GRADCAM_AVAILABLE else "manual fallback"}')
print(f'Target layer: {type(grad_cam.target_layer).__name__}')
dummy_view=torch.randn(1,3,224,224).to(DEVICE)
heatmap=grad_cam.generate_heatmap(dummy_view,target_class=1)
print(f'Heatmap shape={heatmap.shape} range=[{heatmap.min():.3f},{heatmap.max():.3f}]')
fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Grad-CAM - AI Decision Heatmap for Radiologist Review',fontweight='bold')
raw=dummy_view[0].permute(1,2,0).cpu().numpy()
raw=(raw-raw.min())/(raw.max()-raw.min()+1e-8)
raw_u8=(raw*255).astype(np.uint8)
if raw_u8.ndim==2: raw_u8=np.stack([raw_u8]*3,axis=-1)
axes[0].imshow(raw,cmap='gray'); axes[0].set_title('Input Mammogram View'); axes[0].axis('off')
axes[1].imshow(heatmap,cmap='hot'); axes[1].set_title('Grad-CAM Heatmap'); axes[1].axis('off')
overlay=grad_cam.overlay(raw_u8,heatmap)
axes[2].imshow(overlay); axes[2].set_title('Overlay (for radiologist)'); axes[2].axis('off')
plt.tight_layout(); plt.savefig('/tmp/gradcam.png',dpi=120,bbox_inches='tight'); plt.show()
print('Grad-CAM complete')


In [ ]:
# ─────────────────────────────────────────────────────────────────
# FULL END-TO-END PIPELINE
# Mirrors AIMS system architecture:
#   PACS → AI exclusion check → 4-view preprocessing →
#   Density model → Lesion detection → BIRADS scoring →
#   Risk score → Time-point prediction → HL7 report
# AI read latency in AIMS: 17.7 min median (vs 2.08 days human R1)
# ─────────────────────────────────────────────────────────────────

@dataclass
class AIScreeningResult:
    """Complete AI screening result for one case."""
    patient_id         : str
    study_uid          : str
    manufacturer       : str

    # Density
    density_birads     : int
    density_label      : str

    # Cancer risk
    cancer_prob        : float
    risk_score         : float
    birads_score       : int

    # Decision
    recall_recommended : bool
    threshold_used     : float

    # Lesion
    lesion_type        : str
    invasive_prob      : float
    lesion_size_mm_est : float
    rois               : List[dict]

    # Future risk
    future_risk_39m    : float
    predicted_timepoint: str

    # Metadata
    processing_time_s  : float
    eligible           : bool
    exclusion_reason   : str = ''
    timestamp          : str = field(default_factory=lambda: datetime.now().isoformat())


class AIMSInferencePipeline:
    """
    Complete inference pipeline replicating AIMS AI system workflow.
    """

    def __init__(self,
                 pacs_client: PACSClient,
                 hl7_messenger: HL7Messenger,
                 density_model: nn.Module,
                 birads_model: nn.Module,
                 lesion_model: nn.Module,
                 ic_predictor: IntervalCancerPredictor,
                 operating_point: float = 0.50,
                 device: torch.device = DEVICE):
        self.pacs      = pacs_client
        self.hl7       = hl7_messenger
        self.density_m = density_model.eval()
        self.birads_m  = birads_model.eval()
        self.lesion_m  = lesion_model.eval()
        self.ic_pred   = ic_predictor
        self.op        = operating_point
        self.device    = device
        self.prep      = MammographyPreprocessor()
        self.loc       = ROILocalizationMetrics()

    @torch.no_grad()
    def run(self, dicom_list: List[dict],
            case_metadata: dict = None) -> AIScreeningResult:
        t0 = time.time()
        meta = case_metadata or {}
        pid  = dicom_list[0]['patient_id'] if dicom_list else 'UNKNOWN'
        uid  = dicom_list[0]['study_uid']  if dicom_list else ''
        mfr  = dicom_list[0]['manufacturer'] if dicom_list else 'Unknown'

        # ── Step 1: Eligibility check ──────────────────────────
        eligible, reason = self.pacs.validate_case(dicom_list)
        if not eligible:
            return AIScreeningResult(
                patient_id=pid, study_uid=uid, manufacturer=mfr,
                density_birads=0, density_label='N/A (excluded)',
                cancer_prob=0.0, risk_score=0.0, birads_score=0,
                recall_recommended=False, threshold_used=self.op,
                lesion_type='N/A', invasive_prob=0.0, lesion_size_mm_est=0.0,
                rois=[], future_risk_39m=0.0,
                predicted_timepoint='excluded',
                processing_time_s=time.time()-t0,
                eligible=False, exclusion_reason=reason
            )

        # ── Step 2: Preprocess images ─────────────────────────
        imgs_np = self.prep.preprocess_case(dicom_list)  # (4,3,H,W)
        imgs_t  = torch.tensor(imgs_np).unsqueeze(0).to(self.device)  # (1,4,3,H,W)

        # ── Step 3: Density classification ────────────────────
        d_out         = self.density_m(imgs_t)
        density_class = int(d_out['pred'][0].item())
        density_label = DensityClassifier.BIRADS_LABELS[density_class]

        # ── Step 4: Full BIRADS scoring ────────────────────────
        meta_vec = torch.zeros(1, 8).to(self.device)
        meta_vec[0, 0] = (meta.get('age', 60) - 50) / 20.0
        meta_vec[0, 1] = density_class / 3.0
        b_out = self.birads_m(imgs_t, meta_vec)

        cancer_prob = float(b_out['cancer_prob'][0].item())
        birads_pred = int(b_out['birads_pred'][0].item())
        recall      = cancer_prob >= self.op

        # ── Step 5: Lesion classification on central ROI ───────
        roi_img  = imgs_t[0, 0, :, :, :]  # First view CC_LEFT
        l_out    = self.lesion_m(roi_img.unsqueeze(0))
        lesion_type_idx = int(l_out['lesion_type_logits'].argmax(1)[0].item())
        lesion_type     = LesionClassifier.LESION_TYPES[lesion_type_idx]
        invasive_prob   = float(l_out['invasive_prob'][0].item())
        size_est        = float(l_out['size_mm_pred'][0].item())

        # Generate ROI (mock bounding box)
        rois = []
        if recall:
            rois = [{
                'laterality': 'L', 'view': 'MLO',
                'x': 400, 'y': 300, 'w': 80, 'h': 60,
                'confidence': cancer_prob,
                'lesion_type': lesion_type
            }]

        # ── Step 6: Future cancer risk ─────────────────────────
        future = self.ic_pred.predict_timepoint(
            cancer_prob=cancer_prob,
            density_birads=density_class+1,
            age=meta.get('age', 60),
            screen_type=meta.get('screen_type', 'incident'),
            roi_confidence=cancer_prob if recall else 0.0
        )

        processing_time = time.time() - t0

        result = AIScreeningResult(
            patient_id=pid, study_uid=uid, manufacturer=mfr,
            density_birads=density_class+1, density_label=density_label,
            cancer_prob=cancer_prob, risk_score=cancer_prob,
            birads_score=birads_pred,
            recall_recommended=recall, threshold_used=self.op,
            lesion_type=lesion_type, invasive_prob=invasive_prob,
            lesion_size_mm_est=size_est, rois=rois,
            future_risk_39m=future['future_risk_39m'],
            predicted_timepoint=future['predicted_timepoint'],
            processing_time_s=processing_time,
            eligible=True, exclusion_reason=''
        )
        return result

    def generate_report(self, result: AIScreeningResult,
                         export_hl7: bool = True) -> dict:
        """Generate clinical report + HL7 / FHIR messages."""
        report = {
            'report_timestamp' : result.timestamp,
            'patient_id'       : result.patient_id,
            'study_uid'        : result.study_uid,
            'manufacturer'     : result.manufacturer,
            'eligible'         : result.eligible,
            'exclusion_reason' : result.exclusion_reason,
            'density' : {
                'birads'       : result.density_birads,
                'label'        : result.density_label,
            },
            'cancer_assessment': {
                'risk_score'   : round(result.risk_score, 4),
                'birads'       : result.birads_score,
                'recall'       : result.recall_recommended,
                'threshold'    : result.threshold_used,
            },
            'lesion': {
                'type'         : result.lesion_type,
                'invasive_prob': round(result.invasive_prob, 3),
                'size_mm_est'  : round(result.lesion_size_mm_est, 1),
                'rois'         : result.rois,
            },
            'future_risk': {
                '39m_risk'     : round(result.future_risk_39m, 4),
                'predicted_tp' : result.predicted_timepoint,
                'early_recall' : result.future_risk_39m > 0.15,
            },
            'performance': {
                'processing_time_s' : round(result.processing_time_s, 3),
                'note'         : 'AIMS median: 17.7 min vs 2.08 days (R1)'
            }
        }

        if export_hl7 and result.eligible:
            ai_result_dict = {
                'birads'        : result.birads_score,
                'recall'        : result.recall_recommended,
                'risk_score'    : result.risk_score,
                'density_birads': f'B{result.density_birads}',
                'rois'          : result.rois,
            }
            report['hl7_message'] = self.hl7.build_oru_r01(
                result.patient_id, result.study_uid, ai_result_dict)
            report['fhir_report'] = self.hl7.build_fhir_diagnostic_report(
                result.patient_id, ai_result_dict)

        return report


# ─── Run the full pipeline ────────────────────────────────────────
# FIX: ensure threshold is defined even if cells ran out of order
if 'op' not in dir(): op={'threshold':0.5,'sensitivity':0.5,'specificity':0.9,'recall_rate':0.065,'cdr_per_1k':8.0}
threshold = op['threshold']
if 'grad_cam' not in dir(): grad_cam=MammographyGradCAM(birads_model)
if 'yolo_detector' not in dir(): yolo_detector=YOLOLesionDetector()

pipeline = AIMSInferencePipeline(
    pacs_client    = pacs,
    hl7_messenger  = hl7,
    density_model  = density_model,
    birads_model   = birads_model,
    lesion_model   = lesion_model,
    ic_predictor   = ic_predictor,
    operating_point= threshold,
    device         = DEVICE
)

# Run on mock case
result = pipeline.run(
    mock_case,
    case_metadata={'age': 58, 'screen_type': 'incident', 'imd': 5}
)
report = pipeline.generate_report(result, export_hl7=True)

print('='*60)
print('🏥 AIMS AI SCREENING REPORT')
print('='*60)
print(f"Patient       : {report['patient_id']}")
print(f"Eligible      : {report['eligible']}")
print(f"Density       : BIRADS-{report['density']['birads']} — {report['density']['label']}")
print(f"Cancer Risk   : {report['cancer_assessment']['risk_score']:.4f}")
print(f"BIRADS Score  : {report['cancer_assessment']['birads']}")
print(f"RECALL?       : {'YES ⚠️' if report['cancer_assessment']['recall'] else 'NO ✅'}")
print(f"Lesion Type   : {report['lesion']['type']}")
print(f"Invasive Prob : {report['lesion']['invasive_prob']:.3f}")
print(f"Size Estimate : {report['lesion']['size_mm_est']:.1f} mm")
print(f"39m Risk      : {report['future_risk']['39m_risk']:.4f}")
print(f"Time-point    : {report['future_risk']['predicted_tp']}")
print(f"Processing    : {report['performance']['processing_time_s']:.3f}s")
print(f"              → {report['performance']['note']}")
print('='*60)
if 'hl7_message' in report:
    print('\n📨 HL7 Message (first 300 chars):')
    print(report['hl7_message'][:300])

## 🌡️ Section S10 — Probability Calibration (Temperature & Platt Scaling)

In [ ]:
# S10 - PROBABILITY CALIBRATION: Temperature Scaling + Platt Scaling
# Fixes overconfident predictions. Required for reliable OP selection.

class TemperatureScaling(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)
    def forward(self, logits):
        return torch.sigmoid(logits / self.temperature.clamp(min=0.05))
    def fit(self, logits, labels, n_iter=500, lr=0.01):
        optimizer = optim.LBFGS([self.temperature], lr=lr, max_iter=n_iter)
        nll = nn.BCEWithLogitsLoss()
        def closure():
            optimizer.zero_grad()
            loss = nll(logits / self.temperature.clamp(min=0.05), labels)
            loss.backward()
            return loss
        optimizer.step(closure)
        return float(self.temperature.item())


class PlattScaling:
    def __init__(self):
        from sklearn.linear_model import LogisticRegression
        self.lr_model = LogisticRegression(C=1.0, solver='lbfgs', max_iter=1000)
        self.fitted = False
    def fit(self, scores, labels):
        self.lr_model.fit(scores.reshape(-1, 1), labels)
        self.fitted = True
        print(f"  Platt: A={self.lr_model.coef_[0,0]:.4f}  B={self.lr_model.intercept_[0]:.4f}")
    def predict_proba(self, scores):
        assert self.fitted
        return self.lr_model.predict_proba(scores.reshape(-1, 1))[:, 1]


def plot_calibration(probs_raw, probs_cal, labels, title="Calibration"):
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title, fontweight='bold')
    for ax, probs, name, color in [
        (axes[0], probs_raw, 'Before calibration', '#E74C3C'),
        (axes[1], probs_cal, 'After calibration',  '#27AE60'),
    ]:
        frac_pos, mean_pred = calibration_curve(labels, probs, n_bins=10)
        ax.plot(mean_pred, frac_pos, 'o-', color=color, label=name, lw=2)
        ax.plot([0,1],[0,1],'k--', label='Perfect', alpha=0.5)
        ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
        ax.set_title(name); ax.legend(); ax.grid(True, alpha=0.3)
        n = len(probs)
        hist_counts = np.histogram(probs, bins=10)[0][:len(frac_pos)]
        ece = float(np.sum(np.abs(frac_pos - mean_pred) * hist_counts) / n)
        ax.text(0.05, 0.92, f'ECE={ece:.4f}', transform=ax.transAxes, fontsize=11,
                bbox=dict(facecolor='white', alpha=0.8))
    plt.tight_layout(); plt.show()


rng_cal = np.random.default_rng(99)
n_cal = 500
labels_cal   = (rng_cal.random(n_cal) < 0.017).astype(int)
logits_cal   = torch.from_numpy(rng_cal.normal(0, 2, n_cal).astype(np.float32))
labels_cal_t = torch.from_numpy(labels_cal.astype(np.float32))
probs_raw_cal = torch.sigmoid(logits_cal).numpy()

ts  = TemperatureScaling()
T   = ts.fit(logits_cal, labels_cal_t)
probs_ts = ts(logits_cal).detach().numpy()
ps  = PlattScaling()
ps.fit(probs_raw_cal, labels_cal)
probs_platt = ps.predict_proba(probs_raw_cal)

print(f"S10 Temperature T = {T:.4f}")
print(f"  Brier raw   = {brier_score_loss(labels_cal, probs_raw_cal):.4f}")
print(f"  Brier temp  = {brier_score_loss(labels_cal, probs_ts):.4f}")
print(f"  Brier Platt = {brier_score_loss(labels_cal, probs_platt):.4f}")
plot_calibration(probs_raw_cal, probs_ts, labels_cal, "S10 Calibration")
print("S10 complete: TemperatureScaling + PlattScaling ready")


## 🔍 Section S12 — Hyperparameter Optimisation (Optuna)

In [ ]:
# S12 - HYPERPARAMETER OPTIMISATION (OPTUNA)
# TPE sampler + MedianPruner. Use n_trials=50+ for real training.
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_AVAILABLE = True
    print(f"Optuna {optuna.__version__} available")
except ImportError:
    OPTUNA_AVAILABLE = False
    print("WARNING: optuna not installed")

if OPTUNA_AVAILABLE:
    def objective(trial):
        lr              = trial.suggest_float('lr', 1e-5, 1e-3, log=True)
        dropout         = trial.suggest_float('dropout', 0.2, 0.5)
        label_smoothing = trial.suggest_float('label_smoothing', 0.0, 0.1)
        batch_size      = trial.suggest_categorical('batch_size', [8, 16, 32])
        w_effnet        = trial.suggest_float('w_effnet', 0.15, 0.40)
        w_densenet      = trial.suggest_float('w_densenet', 0.15, 0.35)
        w_vit           = trial.suggest_float('w_vit', 0.15, 0.35)
        w_monai         = 1.0 - w_effnet - w_densenet - w_vit
        if w_monai < 0.05:
            raise optuna.exceptions.TrialPruned()
        rng      = np.random.default_rng(trial.number + 42)
        base_auc = 0.82
        lr_b     = -abs(np.log10(lr) + 3.7) * 0.03
        dr_b     = -(dropout - 0.35)**2 * 2
        ls_b     = -(label_smoothing - 0.05)**2 * 5
        auc      = float(np.clip(base_auc + lr_b + dr_b + ls_b + rng.normal(0, 0.015), 0.5, 1.0))
        trial.report(auc, step=0)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
        return auc

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
        study_name='breast_cancer_hpo'
    )
    print("Running Optuna HPO (10 demo trials)...")
    study.optimize(objective, n_trials=10, show_progress_bar=False)
    best = study.best_trial
    print(f"\nBest AUC: {best.value:.4f}")
    for k, v in best.params.items():
        print(f"  {k:20s}: {v}")
    w_m = 1 - best.params['w_effnet'] - best.params['w_densenet'] - best.params['w_vit']
    print(f"  {'w_monai':20s}: {w_m:.4f}")
    try:
        imps = optuna.importance.get_param_importances(study)
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(list(imps.keys()), list(imps.values()), color='#3498DB')
        ax.set_xlabel('Importance'); ax.set_title('S12 Hyperparameter Importance')
        ax.grid(True, alpha=0.3, axis='x'); plt.tight_layout(); plt.show()
    except Exception:
        pass
    BEST_PARAMS = best.params
    BEST_PARAMS['w_monai'] = w_m
else:
    BEST_PARAMS = {'lr':1e-4,'dropout':0.35,'label_smoothing':0.05,
                   'batch_size':16,'w_effnet':0.30,'w_densenet':0.25,
                   'w_vit':0.25,'w_monai':0.20}
print("S12 complete: BEST_PARAMS ready")


## 🚀 Section S13 — Fast Inference: TorchScript + ONNX Export

In [ ]:
# S13 - FAST INFERENCE: TorchScript + ONNX EXPORT  (target < 300 ms/case)
import time

class InferenceWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, images: torch.Tensor) -> torch.Tensor:
        B = images.shape[0]
        meta = torch.zeros(B, 8, device=images.device)
        return self.model(images, meta)['cancer_prob']


def export_torchscript(model, save_path='breast_ai_scripted.pt', input_shape=(1,4,3,224,224)):
    model.eval()
    wrapper = InferenceWrapper(model)
    dummy   = torch.randn(*input_shape).to(DEVICE)
    try:
        scripted = torch.jit.trace(wrapper, dummy)
        torch.jit.save(scripted, save_path)
        times = []
        with torch.no_grad():
            for _ in range(25):
                t0 = time.perf_counter()
                _ = scripted(dummy)
                if DEVICE.type == 'cuda': torch.cuda.synchronize()
                times.append((time.perf_counter()-t0)*1000)
        ms = float(np.mean(times[5:]))
        print(f"  TorchScript: {ms:.1f} ms/case  {'OK < 300ms' if ms < 300 else 'WARNING > 300ms'}")
        return ms
    except Exception as e:
        print(f"  TorchScript failed: {e}"); return -1.0


def export_onnx(model, save_path='breast_ai.onnx', input_shape=(1,4,3,224,224)):
    try:
        import onnx, onnxruntime as ort
    except ImportError:
        print("  ONNX/ONNXRuntime not installed"); return False
    model.eval()
    wrapper = InferenceWrapper(model)
    dummy   = torch.randn(*input_shape).to(DEVICE)
    try:
        with torch.no_grad():
            torch.onnx.export(wrapper, dummy, save_path,
                export_params=True, opset_version=17, do_constant_folding=True,
                input_names=['images'], output_names=['cancer_prob'],
                dynamic_axes={'images':{0:'batch'},'cancer_prob':{0:'batch'}},
                verbose=False)
        onnx.checker.check_model(onnx.load(save_path))
        sess = ort.InferenceSession(save_path,
               providers=['CUDAExecutionProvider','CPUExecutionProvider'])
        dummy_np = dummy.cpu().numpy()
        times = []
        for _ in range(20):
            t0 = time.perf_counter()
            _ = sess.run(None, {'images': dummy_np})
            times.append((time.perf_counter()-t0)*1000)
        print(f"  ONNX Runtime: {float(np.mean(times[5:])):.1f} ms/case")
        return True
    except Exception as e:
        print(f"  ONNX export failed: {e}"); return False


print("S13 Model Export Summary")
for m, n in [(birads_model,'EnsembleClassifier'),(density_model,'DensityClassifier'),
             (lesion_model,'LesionClassifier'),(calc_model,'CalcPatchClassifier')]:
    p = count_params(m)
    s = sum(par.numel()*par.element_size() for par in m.parameters())/1e6
    print(f"  {n:30s}  params={p:>10,}  size={s:.1f}MB")
birads_model.eval()
export_torchscript(birads_model)
export_onnx(birads_model)
print("S13 complete: Load with torch.jit.load() or ort.InferenceSession()")


## 🫁 Section S15 — Breast Region Segmentation (U-Net)

In [ ]:
# S15 - BREAST REGION SEGMENTATION (Lightweight U-Net)
# Removes background artifacts: wedge markers, text overlays, borders.
# Input: (B,1,H,W) grayscale | Output: (B,1,H,W) binary mask

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class BreastUNet(nn.Module):
    CHS = [64, 128, 256, 512]
    def __init__(self):
        super().__init__()
        C = self.CHS
        self.enc1=DoubleConv(1,C[0]); self.enc2=DoubleConv(C[0],C[1])
        self.enc3=DoubleConv(C[1],C[2]); self.enc4=DoubleConv(C[2],C[3])
        self.pool=nn.MaxPool2d(2)
        self.bot =DoubleConv(C[3],C[3]*2)
        self.up4=nn.ConvTranspose2d(C[3]*2,C[3],2,stride=2); self.dec4=DoubleConv(C[3]*2,C[3])
        self.up3=nn.ConvTranspose2d(C[3],C[2],2,stride=2);   self.dec3=DoubleConv(C[2]*2,C[2])
        self.up2=nn.ConvTranspose2d(C[2],C[1],2,stride=2);   self.dec2=DoubleConv(C[1]*2,C[1])
        self.up1=nn.ConvTranspose2d(C[1],C[0],2,stride=2);   self.dec1=DoubleConv(C[0]*2,C[0])
        self.out=nn.Conv2d(C[0],1,1)
    def forward(self, x):
        e1=self.enc1(x); e2=self.enc2(self.pool(e1))
        e3=self.enc3(self.pool(e2)); e4=self.enc4(self.pool(e3))
        b=self.bot(self.pool(e4))
        d4=self.dec4(torch.cat([self.up4(b),e4],1))
        d3=self.dec3(torch.cat([self.up3(d4),e3],1))
        d2=self.dec2(torch.cat([self.up2(d3),e2],1))
        d1=self.dec1(torch.cat([self.up1(d2),e1],1))
        mask=torch.sigmoid(self.out(d1))
        return {'mask':mask}


class BreastSegPipeline:
    SEG_SIZE = 256
    def __init__(self, model, threshold=0.5, device=DEVICE):
        self.model=model.eval().to(device); self.thr=threshold; self.device=device
    @torch.no_grad()
    def segment(self, img_f32):
        h,w=img_f32.shape[:2]
        sm=cv2.resize(img_f32,(self.SEG_SIZE,self.SEG_SIZE))
        t=torch.from_numpy(sm).unsqueeze(0).unsqueeze(0).to(self.device)
        ms=(self.model(t)['mask'][0,0]>self.thr).cpu().numpy().astype(np.uint8)
        k=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5))
        ms=cv2.morphologyEx(ms,cv2.MORPH_CLOSE,k,iterations=3)
        ms=cv2.morphologyEx(ms,cv2.MORPH_OPEN,k,iterations=2)
        mf=cv2.resize(ms.astype(np.float32),(w,h),interpolation=cv2.INTER_NEAREST).astype(bool)
        return {'mask':mf,'masked_image':img_f32*mf,'breast_fraction':float(mf.mean())}
    def visualise(self, img, res):
        fig,axes=plt.subplots(1,3,figsize=(15,5))
        fig.suptitle('S15 Breast Segmentation',fontweight='bold')
        axes[0].imshow(img,cmap='gray'); axes[0].set_title('Original (CLAHE)')
        axes[1].imshow(res['mask'],cmap='gray'); axes[1].set_title('Mask')
        axes[2].imshow(res['masked_image'],cmap='gray')
        axes[2].set_title(f"Masked ({res['breast_fraction']*100:.1f}% breast)")
        [ax.axis('off') for ax in axes]; plt.tight_layout(); plt.show()


breast_unet  = BreastUNet().to(DEVICE)
seg_pipeline = BreastSegPipeline(breast_unet)
print(f"BreastUNet params: {count_params(breast_unet):,}")
rng_seg = np.random.default_rng(7)
dmam = rng_seg.random((512,512)).astype(np.float32)
Y,X  = np.ogrid[:512,:512]
dmam[((Y-256)/220)**2+((X-300)/180)**2<=1] += 0.4
dmam = np.clip(dmam,0,1)
sr   = seg_pipeline.segment(dmam)
print(f"Breast fraction: {sr['breast_fraction']*100:.1f}%")
seg_pipeline.visualise(dmam,sr)
print("S15 complete: BreastUNet ready. Train on INbreast XML contours as GT masks.")


## 🔗 Section S16 — Multi-View Consistency Loss

In [ ]:
# S16 - MULTI-VIEW CONSISTENCY LOSS
# View order: [CC_LEFT(0), CC_RIGHT(1), MLO_LEFT(2), MLO_RIGHT(3)]
# Pairs: 0<->2 (left breast), 1<->3 (right breast)
# Improves ICC by penalising inconsistent cross-view embeddings.

class MultiViewConsistencyLoss(nn.Module):
    def __init__(self, weight=0.10):
        super().__init__()
        self.weight=weight; self.mse=nn.MSELoss()
    def forward(self, embeddings):
        emb=F.normalize(embeddings, p=2, dim=-1)
        loss_left  = self.mse(emb[:,0,:], emb[:,2,:])
        loss_right = self.mse(emb[:,1,:], emb[:,3,:])
        return self.weight*(loss_left+loss_right)/2.0


cl=MultiViewConsistencyLoss(weight=0.10)
# Correlated views (good: CC and MLO of same breast should be similar)
emb_corr=torch.randn(4,4,512)
emb_corr[:,2,:]=emb_corr[:,0,:]+0.1*torch.randn(4,512)
emb_corr[:,3,:]=emb_corr[:,1,:]+0.1*torch.randn(4,512)
emb_rand=torch.randn(4,4,512)
print(f"S16 Multi-View Consistency Loss")
print(f"  Correlated views (good): {cl(emb_corr).item():.6f}")
print(f"  Random views     (bad) : {cl(emb_rand).item():.6f}  <- should be larger")
print()
print("Add to training loss: total = task_loss + MultiViewConsistencyLoss()(view_embeddings)")
print("S16 complete")


## ⛏️ Section S17 — Hard Negative Mining

In [ ]:
# S17 - HARD NEGATIVE MINING
# Elevates sampling frequency of misclassified benign/malignant cases.
# Reduces false positives in BI-RADS 4/5 predictions.
import heapq, copy

class HardNegativeMiner:
    def __init__(self, n_samples, hard_ratio=0.15, weight_multiplier=3.0):
        self.n=n_samples; self.ratio=hard_ratio; self.mult=weight_multiplier
        self.losses=np.ones(n_samples,dtype=np.float32)
        self.weights=np.ones(n_samples,dtype=np.float32)
        self._epoch={}
    def update(self, indices, losses):
        for idx,loss in zip(indices,losses):
            self._epoch.setdefault(idx,[]).append(float(loss))
    def end_epoch(self):
        for idx,ls in self._epoch.items():
            self.losses[idx]=float(np.mean(ls))
        self._epoch.clear()
        thr=np.percentile(self.losses,100*(1-self.ratio))
        self.weights=np.where(self.losses>=thr,self.mult,1.0).astype(np.float32)
        return self.weights
    def sampler(self):
        return WeightedRandomSampler(self.weights,self.n,replacement=True)
    def stats(self):
        n_h=int((self.weights>1.0).sum())
        return {'n_hard':n_h,'pct_hard':n_h/self.n*100,
                'mean_loss_hard':float(self.losses[self.weights>1.0].mean()),
                'mean_loss_easy':float(self.losses[self.weights<=1.0].mean())}


N_TR=1400; miner=HardNegativeMiner(N_TR,hard_ratio=0.15,weight_multiplier=3.0)
rng_h=np.random.default_rng(55)
print("S17 Hard Negative Mining Demo (3 simulated epochs)")
for ep in range(1,4):
    for bs in range(0,N_TR,32):
        bi=list(range(bs,min(bs+32,N_TR)))
        bl=[rng_h.uniform(0.5,1.5) if 300<=i<450 else rng_h.uniform(0.05,0.4) for i in bi]
        miner.update(bi,bl)
    w=miner.end_epoch(); s=miner.stats()
    print(f"  Epoch {ep}: hard={s['n_hard']} ({s['pct_hard']:.1f}%)  "
          f"loss_hard={s['mean_loss_hard']:.3f}  loss_easy={s['mean_loss_easy']:.3f}")

fig,ax=plt.subplots(figsize=(10,4))
ax.scatter(range(N_TR),miner.losses,c=miner.weights,cmap='RdYlGn_r',s=8,alpha=0.7)
thr85=np.percentile(miner.losses,85)
ax.axhspan(thr85,miner.losses.max(),alpha=0.1,color='red',label='Hard zone (top 15%)')
ax.set_xlabel('Sample index'); ax.set_ylabel('Loss')
ax.set_title('S17 Hard Negative Loss Distribution')
ax.legend(); ax.grid(True,alpha=0.3); plt.tight_layout(); plt.show()
print("S17 complete: HardNegativeMiner reduces false positives in BI-RADS 4/5")


## 🔭 Section S18 — Multi-Scale Learning (FPN)

In [ ]:
# S18 - MULTI-SCALE LEARNING (Feature Pyramid Network)
# FPN aggregates multi-scale features from ResNet-50 C3/C4/C5.
# Random scale augmentation: 256/384/512 per batch.

class FPNNeck(nn.Module):
    FPN_CH = 256
    def __init__(self, in_channels=None):
        super().__init__()
        ic=in_channels or [512,1024,2048]; FC=self.FPN_CH
        self.lat3=nn.Conv2d(ic[0],FC,1); self.lat4=nn.Conv2d(ic[1],FC,1); self.lat5=nn.Conv2d(ic[2],FC,1)
        self.sm3=nn.Conv2d(FC,FC,3,padding=1); self.sm4=nn.Conv2d(FC,FC,3,padding=1); self.sm5=nn.Conv2d(FC,FC,3,padding=1)
        self.pool=nn.AdaptiveAvgPool2d(1)
    def forward(self, c3, c4, c5):
        p5=self.lat5(c5)
        p4=self.lat4(c4)+F.interpolate(p5,scale_factor=2,mode='nearest')
        p3=self.lat3(c3)+F.interpolate(p4,scale_factor=2,mode='nearest')
        v5=self.pool(self.sm5(p5)).flatten(1)
        v4=self.pool(self.sm4(p4)).flatten(1)
        v3=self.pool(self.sm3(p3)).flatten(1)
        return torch.cat([v3,v4,v5],dim=1)


class MultiScaleClassifier(nn.Module):
    SCALES=[256,384,512]
    def __init__(self, pretrained=False):
        super().__init__()
        base=_resnet50(pretrained)
        self.layer0=nn.Sequential(base.conv1,base.bn1,base.relu,base.maxpool)
        self.layer1=base.layer1; self.layer2=base.layer2
        self.layer3=base.layer3; self.layer4=base.layer4
        self.fpn=FPNNeck([512,1024,2048])
        self.head=nn.Sequential(nn.Linear(768,256),nn.GELU(),nn.Dropout(0.4),nn.Linear(256,1))
    def forward(self, x):
        x=self.layer0(x); x=self.layer1(x)
        c3=self.layer2(x); c4=self.layer3(c3); c5=self.layer4(c4)
        feat=self.fpn(c3,c4,c5)
        logit=self.head(feat).squeeze(1)
        return {'prob':torch.sigmoid(logit),'fpn_feat':feat}
    @staticmethod
    def random_scale(imgs, scales=None):
        t=random.choice(scales or [256,384,512])
        return F.interpolate(imgs,size=(t,t),mode='bilinear',align_corners=False)


ms_model=MultiScaleClassifier(pretrained=False).to(DEVICE)
print(f"S18 MultiScaleClassifier params: {count_params(ms_model):,}")
for sc in [256,384,512]:
    d=torch.randn(2,3,sc,sc).to(DEVICE)
    with torch.no_grad(): o=ms_model(d)
    print(f"  Scale {sc}x{sc}: FPN={tuple(o['fpn_feat'].shape)}  prob={[round(float(p),3) for p in o['prob'].tolist()]}")
print("Training tip: imgs_scaled = MultiScaleClassifier.random_scale(imgs, [256,384,512])")
print("S18 complete: FPN multi-scale model ready")


## 🌍 Section S20 — Domain Adaptation

In [ ]:
# S20 - DOMAIN ADAPTATION
# Histogram matching: normalise CBIS-DDSM (film) -> INbreast (FFDM) appearance.
# AdaIN: learned feature-level domain alignment inside backbone.

class HistogramMatcher:
    @staticmethod
    def match(source, reference):
        src_u8=(source*255).astype(np.uint8).ravel()
        ref_u8=(reference*255).astype(np.uint8).ravel()
        sh=np.bincount(src_u8,minlength=256)/src_u8.size
        rh=np.bincount(ref_u8,minlength=256)/ref_u8.size
        sc=np.cumsum(sh); rc=np.cumsum(rh)
        lut=np.zeros(256,dtype=np.uint8); j=0
        for i in range(256):
            while j<255 and rc[j]<sc[i]: j+=1
            lut[i]=j
        return lut[(source*255).astype(np.uint8)].astype(np.float32)/255.0


class AdaptiveInstanceNorm(nn.Module):
    def __init__(self, num_features, eps=1e-5):
        super().__init__()
        self.eps=eps; self.style_fc=nn.Linear(num_features,num_features*2)
    def forward(self, content, style_vec):
        B,C,H,W=content.shape
        cm=content.mean(dim=[2,3],keepdim=True)
        cs=content.std(dim=[2,3],keepdim=True).clamp(min=self.eps)
        norm=(content-cm)/cs
        sp=self.style_fc(style_vec)
        gamma,beta=sp.chunk(2,dim=1)
        return gamma.view(B,C,1,1)*norm+beta.view(B,C,1,1)


rng_da=np.random.default_rng(21)
ref_img=np.clip(rng_da.normal(0.55,0.18,(512,512)),0,1).astype(np.float32)
src_img=np.clip(rng_da.normal(0.35,0.22,(512,512)),0,1).astype(np.float32)
matched=HistogramMatcher.match(src_img,ref_img)

fig,axes=plt.subplots(1,3,figsize=(15,4))
fig.suptitle('S20 Domain Adaptation: Histogram Matching',fontweight='bold')
for ax,img,title,color in [
    (axes[0],src_img,'CBIS-DDSM (source)','#E74C3C'),
    (axes[1],ref_img,'INbreast (reference)','#27AE60'),
    (axes[2],matched,'CBIS-DDSM matched','#3498DB')]:
    ax.hist(img.ravel(),bins=64,color=color,alpha=0.7)
    ax.set_title(title); ax.set_xlabel('Intensity'); ax.grid(True,alpha=0.3)
plt.tight_layout(); plt.show()

adain=AdaptiveInstanceNorm(64).to(DEVICE)
cf=torch.randn(2,64,14,14).to(DEVICE); sv=torch.randn(2,64).to(DEVICE)
of=adain(cf,sv)
print(f"S20 Histogram match: {src_img.shape} source -> matched")
print(f"     AdaIN: content={tuple(cf.shape)} -> {tuple(of.shape)}")
print("S20 complete: HistogramMatcher + AdaptiveInstanceNorm ready")


## ❓ Section S21 — Uncertainty Estimation (MC Dropout)

In [ ]:
# S21 - UNCERTAINTY ESTIMATION (Monte Carlo Dropout)
# N forward passes with dropout active. mean=prediction, std=uncertainty.
# Flag std > 0.15 for radiologist review (interval cancers often high uncertainty).

class MCDropoutInference:
    def __init__(self, model, n_passes=30, device=DEVICE):
        self.model=model; self.n=n_passes; self.device=device
    def _enable_dropout(self):
        for m in self.model.modules():
            if isinstance(m,(nn.Dropout,nn.Dropout2d)): m.train()
    @torch.no_grad()
    def predict(self, images, metadata=None):
        self.model.eval(); self._enable_dropout()
        B=images.shape[0]
        if metadata is None: metadata=torch.zeros(B,8).to(self.device)
        all_p=[]
        for _ in range(self.n):
            out=self.model(images,metadata)
            all_p.append(out['cancer_prob'].cpu().numpy())
        all_p=np.stack(all_p,axis=1)
        mean_p=all_p.mean(1); std_p=all_p.std(1)
        return {'mean_prob':mean_p,'std_prob':std_p,'all_probs':all_p,
                'confidence':np.clip(1-2*std_p,0,1),'flag_review':std_p>0.15}
    def plot(self, res, ids=None):
        B=res['mean_prob'].shape[0]; ids=ids or [f"Case {i+1}" for i in range(B)]
        fig,axes=plt.subplots(1,2,figsize=(14,5))
        fig.suptitle('S21 MC Dropout Uncertainty',fontweight='bold')
        ax=axes[0]; x=np.arange(B)
        ax.bar(x,res['mean_prob'],color='#3498DB',alpha=0.7,label='Mean prob')
        ax.errorbar(x,res['mean_prob'],yerr=res['std_prob'],fmt='none',color='black',capsize=5)
        ax.axhline(0.5,color='red',ls='--',alpha=0.5,label='Threshold')
        for i,f in enumerate(res['flag_review']):
            if f: ax.annotate('!',xy=(i,res['mean_prob'][i]),fontsize=16,ha='center',color='#E74C3C',fontweight='bold')
        ax.set_xticks(x); ax.set_xticklabels(ids,rotation=30,ha='right')
        ax.legend(); ax.set_title('Prediction +/- Uncertainty')
        axes[1].hist(res['all_probs'][0],bins=15,color='#9B59B6',alpha=0.8)
        axes[1].axvline(res['mean_prob'][0],color='blue',lw=2,label='Mean')
        axes[1].set_xlabel('Cancer prob per MC pass'); axes[1].legend()
        axes[1].set_title(f'{ids[0]}: {self.n} MC Dropout passes')
        plt.tight_layout(); plt.show()


mc=MCDropoutInference(birads_model,n_passes=20)
di=torch.randn(4,4,3,224,224).to(DEVICE)
ur=mc.predict(di)
print("S21 MC Dropout Uncertainty Results")
print(f"{'Case':>6}  {'MeanProb':>9}  {'Std':>7}  {'Confidence':>11}  {'Review?'}")
print("-"*55)
for i in range(4):
    f="YES" if ur['flag_review'][i] else "no"
    print(f"{i+1:>6}  {ur['mean_prob'][i]:>9.4f}  {ur['std_prob'][i]:>7.4f}  "
          f"{ur['confidence'][i]:>11.4f}  {f}")
mc.plot(ur)
print("S21 complete: high-uncertainty cases flagged for radiologist review")


## 💾 Section S22 — Checkpoint Averaging (Top-5)

In [ ]:
# S22 - MODEL CHECKPOINT AVERAGING (Top-5)
# Saves top-K best val checkpoints; averages weights at training end.
# Typically gives 1-2% AUC improvement over single best checkpoint.
import heapq, copy

class CheckpointAverager:
    def __init__(self, model, k=5, save_dir='/tmp/ckpts'):
        self.model=model; self.k=k
        self.save_dir=Path(save_dir); self.save_dir.mkdir(parents=True,exist_ok=True)
        self._heap=[]
    def update(self, val_auc, epoch):
        neg=-val_auc
        path=self.save_dir/f'ckpt_ep{epoch:03d}_auc{val_auc:.4f}.pt'
        if len(self._heap)<self.k:
            torch.save(self.model.state_dict(),path)
            heapq.heappush(self._heap,(neg,epoch,str(path)))
            print(f"  [Ckpt] Saved ep{epoch} AUC={val_auc:.4f} ({len(self._heap)}/{self.k})")
            return True
        elif neg<self._heap[0][0]:
            worst=heapq.heappop(self._heap)
            wp=Path(worst[2])
            if wp.exists(): wp.unlink()
            torch.save(self.model.state_dict(),path)
            heapq.heappush(self._heap,(neg,epoch,str(path)))
            print(f"  [Ckpt] Replaced ep{worst[1]} -> ep{epoch} AUC={val_auc:.4f}")
            return True
        return False
    def average_weights(self):
        valid=[p for _,_,p in self._heap if Path(p).exists()]
        print(f"\nAveraging {len(valid)} checkpoints...")
        avg_sd=None
        for p in valid:
            sd=torch.load(p,map_location='cpu')
            if avg_sd is None:
                avg_sd={k:v.float().clone() for k,v in sd.items()}
            else:
                for k in avg_sd: avg_sd[k]+=sd[k].float()
        n=len(valid)
        for k in avg_sd: avg_sd[k]/=n
        m=copy.deepcopy(self.model); m.load_state_dict(avg_sd)
        print(f"Averaged {n} checkpoints -> final model ready")
        return m
    def summary(self):
        print(f"\nTop-{self.k} Checkpoints:")
        for neg,ep,path in sorted(self._heap):
            print(f"  Epoch {ep:3d}  AUC={-neg:.4f}  {Path(path).name}")


avg=CheckpointAverager(birads_model,k=5,save_dir='/tmp/breast_ckpts')
print("S22 Checkpoint Averager Demo (10 simulated epochs)")
sim_aucs=[0.75,0.79,0.82,0.81,0.84,0.83,0.86,0.85,0.87,0.88]
for ep,auc in enumerate(sim_aucs,1): avg.update(auc,ep)
avg.summary()
final_model=avg.average_weights()
print(f"Final averaged model params: {count_params(final_model):,}")
print("S22 complete: CheckpointAverager ready")


## 📋 Appendix — Key Implementation Notes

### Study Benchmarks (Use These to Validate Your Model)

| Metric | AIMS AI | First Reader | Second Reader | Consensus |
|--------|---------|-------------|---------------|----------|
| Case Sensitivity | **0.541** | 0.437 | 0.500 | 0.478 |
| Case Specificity | **0.943** | 0.952 | 0.934 | 0.945 |
| CDR per 1,000 | **9.33** | 7.54 | - | - |
| Recall Rate | 6.5% | 5.5% | - | - |
| Breast AUC | **0.84** | - | - | - |
| AUC (screen-detected only) | **0.978** | - | - | - |
| Interval Cancer sensitivity | **25.0%** | 0% | 0% | 0% |
| Invasive sensitivity | **0.54** | 0.43 | 0.46 | 0.46 |
| First screens recall reduction | **−39.3%** | baseline | - | - |
| Reading time reduction (2nd reader) | **−32%** | - | - | - |

### Warren et al. Companion Study (Arbitration Impact)

| Phase | AI Arm Sensitivity | Human Arm Sensitivity |
|-------|--------------------|----------------------|
| Before arbitration — screen-detected | 99.1% | 100.0% |
| Before arbitration — interval | **32.4%** | 15.4% |
| Before arbitration — next-round | **34.0%** | 12.8% |
| After arbitration — overall | **49.2%** | 48.0% |
| After arbitration — interval | 8.8% | 5.9% |
| Reading workload reduction | **−40%** | baseline |

### Critical Implementation Warnings

1. **Distribution Shift**: 8-year lag caused recall rates to double. Always calibrate on local recent data.
2. **Manufacturer variability**: Hologic Selenia Dimensions vs Lorad → 10.9% vs 6.3% recall. Test per-device.
3. **Fairness monitoring**: 150–200 positive cases per subgroup needed for reliable NI testing.
4. **HL7/NBSS writeback**: NBSS at time of AIMS study lacked AI writeback. Build this explicitly.
5. **Arbitration**: Radiologists overruled 93 correct AI recalls (mostly interval/next-round). Explainability critical.
6. **Operating point**: Never assume same OP across sites. Use site-specific tuning sets with no overlap with test data.
7. **Ground truth**: Use 39-month follow-up window (not episode-only) for comprehensive sensitivity measurement.

### Future Directions (from AIMS Discussion)
- AI triage: high-confidence cases direct to recall, avoiding arbitration
- Incorporate prior images into the model (currently excluded)
- Risk-stratified screening: AI identifies groups for earlier follow-up
- EDITH trial (UK): prospective clinical trial for AI in breast screening
- Improve explainability to reduce arbitration overruling of correct AI recalls